# DataInsight | Time series: Dự báo lưu lượng xe theo giờ trên cao tốc I-94

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trthminh/CO5177/blob/main/notebooks/timeseries.ipynb)

| | |
|---|---|
| **Môn học** | Nền tảng lập trình cho phân tích và trực quan dữ liệu (CO5177), HK 261, 2026–2027 |
| **Giảng viên** | Lê Thành Sách, Khoa Khoa học và Kỹ thuật Máy tính, ĐH Bách Khoa – ĐHQG-HCM |
| **Nhóm** | DataInsight: Trương Thanh Minh (2580893), Tô Anh Phát (2580894), Trần Lê Hoàng Long (2570443) |
| **Phụ trách notebook** | Trương Thanh Minh (2580893) |
| **Dữ liệu** | [Metro Interstate Traffic Volume](https://www.kaggle.com/datasets/pooriamst/metro-interstate-traffic-volume) (MnDOT, UCI), CC BY 4.0 |

## Tóm tắt

Notebook này dự báo số xe đi qua một trạm đếm trên cao tốc I-94 (Minnesota, Mỹ) trong giờ kế tiếp. Dữ liệu thô có dòng trùng giờ, có khoảng trống dài tới mười tháng và có lỗi cảm biến; làm sạch xong còn một chuỗi đều 24 096 giờ. Phân tích khám phá cho thấy chuỗi chủ yếu do chu kỳ ngày và tuần quyết định, phụ thuộc mạnh vào vài giờ vừa qua, phần còn lại có đuôi dày vì các sự kiện bất thường, và bị tuyết cùng ngày lễ tác động rõ. Từ những đặc điểm này chúng tôi chọn đặc trưng, mô hình và metric. Mô hình cuối (HistGradientBoosting) đạt **MAE 130.8 xe/giờ trên test, thấp hơn 49% so với cách lặp lại cùng giờ tuần trước** (mốc mạnh nhất). Mô hình vẫn thấp hơn mốc này 32% khi dự báo trước 24 giờ. Ngoài ra, khi hình dạng tuần được mô tả trực tiếp bằng one-hot giờ trong tuần, một mô hình tuyến tính cũng đạt chất lượng gần như ngang bằng.

## Trình tự phân tích

1. **Bài toán** (mục 2): dự báo cái gì, từ thông tin nào, thuộc loại bài toán nào.
2. **Dữ liệu thô** (mục 3): cấu trúc, chất lượng, phân phối, và các đề xuất tiền xử lý.
3. **Tiền xử lý** (mục 4): làm sạch chuỗi và chia tập theo thời gian.
4. **Phân tích khám phá** (mục 5, chỉ trên train): hình dạng cuối cùng của dữ liệu.
5. **Đặc trưng** (mục 6), **metric và mô hình** (mục 7): mỗi lựa chọn dựa vào một đặc điểm ở mục 5.
6. **Đánh giá** (mục 8) và **kết luận** (mục 9).

Các tham số rút ra từ phân tích (giai đoạn nghiên cứu, chu kỳ, lag, danh sách ngày lễ, …) được khai báo ngay ở ô có bằng chứng cho nó, không khai báo sẵn từ đầu.

**Mục lục:** [1. Thiết lập](#s1) · [2. Bài toán](#s2) · [3. Dữ liệu thô](#s3) · [4. Tiền xử lý](#s4) · [5. Phân tích khám phá](#s5) · [6. Đặc trưng](#s6) · [7. Metric và mô hình](#s7) · [8. Đánh giá](#s8) · [9. Kết luận](#s9)

> **Cách chạy:** mở bằng Colab → **Runtime → Run all** (khoảng 10–15 phút trên Colab). Notebook tự cài thư viện còn thiếu và tự tải dữ liệu.

<a id="s1"></a>
## 1. Thiết lập

In [1]:
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {"numpy": "numpy", "pandas": "pandas>=2.0", "sklearn": "scikit-learn>=1.2",
                     "plotly": "plotly>=5.15", "statsmodels": "statsmodels>=0.14"}
missing = [spec for module, spec in REQUIRED_PACKAGES.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
print("Installed now:", missing or "nothing (all packages present)")

Installed now: nothing (all packages present)


In [2]:
import itertools
import time
import urllib.request
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from statsmodels.tools.sm_exceptions import InterpolationWarning
from statsmodels.tsa.seasonal import MSTL
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import acf, adfuller, kpss, pacf

import sklearn
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", module="statsmodels")            # SARIMAX convergence messages
warnings.filterwarnings("ignore", category=InterpolationWarning)   # KPSS p-values are clipped to [0.01, 0.10]
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.3f}".format)
print(f"numpy {np.__version__} | pandas {pd.__version__} | scikit-learn {sklearn.__version__}")

numpy 2.4.6 | pandas 2.2.3 | scikit-learn 1.6.1


Ở bước này chỉ khai báo những gì đã biết trước khi nhìn dữ liệu: nguồn dữ liệu, tên cột và tầm dự báo.

In [3]:
SEED = 42
np.random.seed(SEED)

DATA_URLS = [   # tried in order: the UCI archive, then the copy in the group's GitHub repository
    "https://archive.ics.uci.edu/ml/machine-learning-databases/00492/Metro_Interstate_Traffic_Volume.csv.gz",
    "https://raw.githubusercontent.com/trthminh/CO5177/main/notebooks/data/Metro_Interstate_Traffic_Volume.csv.gz",
]
DATA_PATH = Path("data") / "Metro_Interstate_Traffic_Volume.csv.gz"
TIME_COL, TARGET, FREQ = "date_time", "traffic_volume", "h"
HORIZON = 1                                                              # forecast one hour ahead (section 2)
NUM_EXOG = ["temp_c", "rain_1h", "snow_1h", "clouds_all", "n_weather"]   # temp_c, n_weather are derived in 3.1
CAT_EXOG = ["weather_main", "weather_description"]

Hình vẽ bằng **Plotly**, rê chuột để xem giá trị. Hàm `show_fig` tự đánh số hình, và tiêu đề mỗi hình ghi luôn kết luận của hình đó. Chữ trong hình viết bằng tiếng Anh.

In [4]:
BLUE, ORANGE, AQUA, YELLOW, RED, MUTED = "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e34948", "#898781"
GRID, INK_2 = "#e1e0d9", "#52514e"
SEQ_BLUE = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]
MONTH_LABELS = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

pio.templates["datainsight"] = go.layout.Template(layout=dict(
    font=dict(family="Inter, Segoe UI, Arial, sans-serif", size=13, color="#0b0b0b"),
    colorway=[BLUE, ORANGE, AQUA, YELLOW, RED, MUTED], title=dict(x=0.01, font=dict(size=16)),
    xaxis=dict(gridcolor=GRID, linecolor="#c9c8c3", zeroline=False, title_font_color=INK_2),
    yaxis=dict(gridcolor=GRID, linecolor="#c9c8c3", zeroline=False, title_font_color=INK_2),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0, title_text=""),
    margin=dict(l=60, r=20, t=80, b=50)))
pio.templates.default = "plotly_white+datainsight"
pio.renderers.default = "colab" if "google.colab" in sys.modules else "plotly_mimetype+notebook"

_figure_number = itertools.count(1)


def show_fig(fig, title, name, height=420, **layout):
    """Number the figure, give it a conclusion-style title and display it (`name` = file name used by the report)."""
    fig.update_layout(title=f"Figure {next(_figure_number)}. {title}", height=height, meta={"name": name}, **layout)
    fig.show()


def hist_bar(values, bins=40, color=BLUE):
    """Histogram computed with NumPy, so the notebook stores ~40 bars instead of every data point."""
    values = np.asarray(values, dtype=float)
    counts, edges = np.histogram(values[~np.isnan(values)], bins=bins)
    return go.Bar(x=(edges[:-1] + edges[1:]) / 2, y=counts, width=np.diff(edges) * 0.92, marker_color=color,
                  hovertemplate="%{x:,.2f}: %{y:,}<extra></extra>")


def band(fig, x, lo, hi, color, row=None, col=None):
    """Filled band between the lines lo and hi."""
    fig.add_trace(go.Scatter(x=np.r_[x, x[::-1]], y=np.r_[hi, lo[::-1]], fill="toself", fillcolor=color,
                             line=dict(width=0), hoverinfo="skip", showlegend=False),
                  **(dict(row=row, col=col) if row else {}))

<a id="s2"></a>
## 2. Bài toán

Sở Giao thông Minnesota (MnDOT) đặt trạm đếm xe ATR 301 trên các làn hướng tây của cao tốc I-94, đoạn giữa Minneapolis và St Paul. Mục tiêu là **dự báo số xe của giờ kế tiếp, khi đang đứng ở cuối giờ hiện tại**, dựa trên lưu lượng đã đo, lịch (giờ, thứ, ngày lễ) và thời tiết. Những dự báo kiểu này được dùng để điều tiết đèn vào cao tốc, xử lý sự cố và báo thời gian di chuyển cho tài xế.

Thứ cần dự báo là một số đếm, không phải nhãn lớp, nên đây là bài toán **hồi quy có giám sát**. Mỗi giờ trong quá khứ cho ta một cặp mẫu: những gì đã biết trước giờ đó, và số xe thật của giờ đó. Khác dữ liệu bảng thông thường, các quan sát ở đây có thứ tự thời gian, và lưu lượng giờ này rất có thể phụ thuộc vào các giờ trước. Vì thế chúng tôi xem nó là bài toán **dự báo chuỗi thời gian một bước**, với lịch và thời tiết là biến ngoại sinh.

Cách đặt bài toán này kéo theo một ràng buộc phải giữ suốt notebook: dự báo cho giờ $t$ **chỉ được dùng thông tin có ở cuối giờ $t-1$**. Còn chuỗi phụ thuộc vào quá khứ đến mức nào, thời tiết và ngày lễ có thật sự tác động không, và nên đo sai số bằng gì, thì các mục sau sẽ trả lời dần từ dữ liệu.

<a id="s3"></a>
## 3. Dữ liệu thô: cấu trúc, chất lượng và phân phối

Trước khi nhìn vào dữ liệu, chúng tôi để riêng **giai đoạn cuối của tệp (từ 12/2017) cho đánh giá**. Khoảng mười tháng này có cả một mùa đông lẫn một mùa hè. Mọi phân phối trong mục này chỉ tính trên phần trước mốc đó, để không có quyết định nào dựa vào chính dữ liệu dùng để chấm điểm. Riêng kiểm tra chất lượng (trùng, thiếu, lỗi đo) thì xét toàn bộ tệp, vì bước này chỉ phát hiện lỗi chứ không chọn gì cho mô hình.

In [5]:
HOLDOUT_START = pd.Timestamp("2017-12-01 00:00")   # data from here on is reserved for validation and test (section 4.2)

### 3.1 Nguồn và cấu trúc

Tệp CSV được tải thẳng từ kho UCI, không cần API key. Mỗi dòng là một giờ (giờ địa phương US Central), gồm 9 cột:

| Cột | Ý nghĩa | Kiểu | Vai trò |
|---|---|---|---|
| `traffic_volume` | số xe trong giờ (xe/giờ) | số đếm, không âm | **nhãn** |
| `date_time` | thời điểm bắt đầu giờ | thời gian | trục thời gian, đặc trưng lịch |
| `holiday` | tên ngày lễ hoặc `None` | phân loại | đầu vào (lịch) |
| `temp` | nhiệt độ (K) | số liên tục | đầu vào (thời tiết) |
| `rain_1h`, `snow_1h` | lượng mưa, tuyết trong giờ (mm) | số | đầu vào (thời tiết) |
| `clouds_all` | độ che phủ mây (%) | số nguyên 0–100 | đầu vào (thời tiết) |
| `weather_main`, `weather_description` | nhóm và mô tả thời tiết | phân loại | đầu vào (thời tiết) |

Khi đọc tệp, chúng tôi tạo thêm hai cột: `temp_c` (nhiệt độ theo °C) và `n_weather` (số dòng có cùng mốc giờ, giải thích ở 3.2).

In [6]:
DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
for url in DATA_URLS:                                        # download once, then reuse the cached file
    if DATA_PATH.exists() and DATA_PATH.stat().st_size > 100_000:
        break
    try:
        urllib.request.urlretrieve(url, DATA_PATH)
        print("downloaded from", url)
    except Exception as exc:                                 # try the next source
        print(f"could not download from {url}: {exc}")
assert DATA_PATH.exists() and DATA_PATH.stat().st_size > 100_000, "data file could not be downloaded from any source"
raw = pd.read_csv(DATA_PATH, keep_default_na=False)          # keep the string "None" of `holiday` as a value
n_raw_cols = raw.shape[1]
raw["timestamp"] = pd.to_datetime(raw[TIME_COL])
raw["temp_c"] = raw["temp"] - 273.15
raw["n_weather"] = raw.groupby("timestamp")["weather_main"].transform("size")

calendar_hours = pd.date_range(raw["timestamp"].min(), raw["timestamp"].max(), freq=FREQ)
print(f"{len(raw):,} rows x {n_raw_cols} columns | {raw['timestamp'].min():%d/%m/%Y %H:%M} – "
      f"{raw['timestamp'].max():%d/%m/%Y %H:%M} | {len(calendar_hours):,} hours in the calendar")
raw.head()

downloaded from https://archive.ics.uci.edu/ml/machine-learning-databases/00492/Metro_Interstate_Traffic_Volume.csv.gz
48,204 rows x 9 columns | 02/10/2012 09:00 – 30/09/2018 23:00 | 52,551 hours in the calendar


,holiday,temp,rain_1h,snow_1h,clouds_all,weather_main,weather_description,date_time,traffic_volume,timestamp,temp_c,n_weather
0,None,288.280,0.000,0.000,40,Clouds,scattered clouds,2012-10-02 09:00:00,5545,2012-10-02 09:00:00,15.130,1
1,None,289.360,0.000,0.000,75,Clouds,broken clouds,2012-10-02 10:00:00,4516,2012-10-02 10:00:00,16.210,1
2,None,289.580,0.000,0.000,90,Clouds,overcast clouds,2012-10-02 11:00:00,4767,2012-10-02 11:00:00,16.430,1
3,None,290.130,0.000,0.000,90,Clouds,overcast clouds,2012-10-02 12:00:00,5026,2012-10-02 12:00:00,16.980,1
4,None,291.140,0.000,0.000,75,Clouds,broken clouds,2012-10-02 13:00:00,4918,2012-10-02 13:00:00,17.990,1


In [7]:
summary_cat = raw[["holiday", *CAT_EXOG]].agg(["nunique", lambda s: s.mode().iloc[0]]).T
summary_cat.columns = ["distinct values", "most frequent"]
display(raw[[TARGET, "temp", "rain_1h", "snow_1h", "clouds_all"]].describe().T)
summary_cat

,count,mean,std,min,25%,50%,75%,max
traffic_volume,"48,204.000","3,259.818","1,986.861",0.000,"1,193.000","3,380.000","4,933.000","7,280.000"
temp,"48,204.000",281.206,13.338,0.000,272.160,282.450,291.806,310.070
rain_1h,"48,204.000",0.334,44.789,0.000,0.000,0.000,0.000,"9,831.300"
snow_1h,"48,204.000",0.000,0.008,0.000,0.000,0.000,0.000,0.510
clouds_all,"48,204.000",49.362,39.016,0.000,1.000,64.000,90.000,100.000


,distinct values,most frequent
holiday,12,None
weather_main,11,Clouds
weather_description,38,sky is clear


Tệp có **48 204 dòng, trong khi giai đoạn này có 52 551 giờ theo lịch**. Chuỗi thời gian cần mỗi giờ đúng một dòng, nên chênh lệch này cho thấy dữ liệu vừa **thiếu giờ** vừa có **giờ bị lặp** (3.2). Bảng thống kê còn có hai giá trị không thể có trong thực tế: nhiệt độ **0 K** và lượng mưa **9 831 mm** trong một giờ (3.3).

### 3.2 Tính đều của trục thời gian

Chúng tôi so từng cột của các dòng trùng mốc giờ để xem chúng khác nhau ở đâu.

In [8]:
dup_mask = raw["timestamp"].duplicated(keep=False)
n_extra = int(raw["timestamp"].duplicated().sum())
cols = ["holiday", TARGET, "temp", "rain_1h", "snow_1h", "clouds_all", *CAT_EXOG]
differs = (raw[dup_mask].groupby("timestamp")[cols].nunique() > 1).mean().mul(100)
print(f"{n_extra:,} extra rows share a timestamp ({n_extra / len(raw):.1%} of rows); "
      f"{raw.loc[dup_mask, 'timestamp'].nunique():,} hours are affected")
display(differs.round(1).rename("% of duplicated hours in which the column differs").to_frame())
raw.loc[raw["timestamp"] == "2013-05-19 10:00", [TIME_COL, TARGET, "temp", *CAT_EXOG, "n_weather"]]

7,629 extra rows share a timestamp (15.8% of rows); 5,445 hours are affected


,% of duplicated hours in which the column differs
holiday,0.000
traffic_volume,0.000
temp,1.400
rain_1h,0.200
snow_1h,0.000
clouds_all,0.600
weather_main,98.200
weather_description,98.900


,date_time,traffic_volume,temp,weather_main,weather_description,n_weather
6162,2013-05-19 10:00:00,3591,287.150,Thunderstorm,thunderstorm with light rain,6
6163,2013-05-19 10:00:00,3591,287.150,Mist,mist,6
6164,2013-05-19 10:00:00,3591,287.150,Thunderstorm,proximity thunderstorm,6
6165,2013-05-19 10:00:00,3591,287.150,Rain,moderate rain,6
6166,2013-05-19 10:00:00,3591,287.150,Rain,light rain,6
6167,2013-05-19 10:00:00,3591,287.150,Thunderstorm,thunderstorm with heavy rain,6


**Có 7 629 dòng (15.8%) trùng mốc giờ.** Ở tất cả các giờ bị trùng, `traffic_volume` đều giống nhau; chỉ `weather_main` và `weather_description` là khác (ở 98% số giờ trùng). Nguồn dữ liệu ghi mỗi hiện tượng thời tiết trong cùng một giờ thành một dòng riêng và chép lại nguyên số xe; ví dụ ở trên là mưa, sương mù và giông lúc 10h ngày 19/5/2013. Gộp các dòng này về một dòng vì vậy không làm sai nhãn. Chúng tôi giữ hiện tượng đầu tiên và lưu số hiện tượng vào `n_weather`.

Sau khi gộp, chuỗi được dựng lại trên lịch đầy đủ để đo các khoảng trống.

In [9]:
def to_regular(df):
    """One row per hour (duplicated hours merged), re-indexed on the full calendar: absent hours become NaN."""
    agg = {TARGET: "mean", **{c: "mean" for c in NUM_EXOG}, **{c: "first" for c in CAT_EXOG}}
    return df.groupby("timestamp").agg(agg).sort_index().asfreq(FREQ)


def gap_table(missing):
    """Start, end and length of every run of consecutive missing hours, longest first."""
    run_id = missing.ne(missing.shift()).cumsum()[missing]
    steps = pd.Series(missing.index[missing], index=run_id.values)
    return (steps.groupby(level=0).agg(start="min", end="max", hours="size")
                 .sort_values("hours", ascending=False).reset_index(drop=True))


regular = to_regular(raw)
is_missing = regular[TARGET].isna()
gaps = gap_table(is_missing)
print(f"{int(is_missing.sum()):,} missing hours ({is_missing.mean():.1%}) in {len(gaps):,} gaps: "
      f"{(gaps['hours'] == 1).mean():.0%} of the gaps last a single hour, but "
      f"{gaps.loc[gaps['hours'] > 168, 'hours'].sum() / gaps['hours'].sum():.0%} of the missing hours come from gaps longer than one week")
gaps.head(3)

11,976 missing hours (22.8%) in 2,588 gaps: 85% of the gaps last a single hour, but 64% of the missing hours come from gaps longer than one week


,start,end,hours
0,2014-08-08 02:00:00,2015-06-11 19:00:00,7386
1,2013-10-27 02:00:00,2013-11-06 03:00:00,242
2,2015-06-14 21:00:00,2015-06-19 17:00:00,117


In [10]:
coverage = (~is_missing).resample("MS").mean() * 100
cov_grid = (coverage.to_frame("pct").assign(year=coverage.index.year, month=coverage.index.month)
                    .pivot(index="year", columns="month", values="pct").reindex(columns=range(1, 13)))
fig = go.Figure(go.Heatmap(z=cov_grid.values, x=MONTH_LABELS, y=cov_grid.index.astype(str),
                           colorscale=[[0, RED], [0.5, "#f6d7c3"], [0.9, "#cde2fb"], [1, BLUE]], zmin=0, zmax=100,
                           text=[["" if np.isnan(v) else f"{v:.0f}" for v in row] for row in cov_grid.values],
                           texttemplate="%{text}", textfont=dict(size=11), xgap=3, ygap=3,
                           colorbar=dict(title="% hours<br>observed"),
                           hovertemplate="%{x} %{y}: %{z:.1f}% of hours observed<extra></extra>"))
show_fig(fig, "Share of hours observed per month: 2014–2015 is mostly empty, 2016–2018 is almost complete",
         "ts_coverage_heatmap", height=380, yaxis_autorange="reversed")

**22.8% số giờ của toàn tệp bị thiếu.** Phần lớn khoảng trống chỉ dài một giờ, nhưng gần hai phần ba số giờ thiếu nằm trong những khoảng trống dài hơn một tuần; khoảng dài nhất là **7 386 giờ liên tiếp (08/2014 – 06/2015)**. Hình 1 cho thấy các khoảng trống dài đều rơi vào trước năm 2016, còn từ 01/2016 tháng nào cũng gần như đủ dữ liệu.

Không có cách điền nào dựng lại được mười tháng dữ liệu, và quanh những lỗ hổng như vậy cũng không có giá trị "cùng giờ tuần trước" để dùng. Do đó chúng tôi chỉ giữ dữ liệu từ 01/2016; bảng dưới kiểm tra lại lựa chọn này.

In [11]:
WINDOW_START, WINDOW_END = "2016-01-01 00:00", "2018-09-30 23:00"   # [derived] the long gaps end in 2015
periods = {"before 2016 (10/2012 – 12/2015)": is_missing[:WINDOW_START].iloc[:-1],
           "study window (01/2016 – 09/2018)": is_missing[WINDOW_START:WINDOW_END]}
window_check = pd.DataFrame({"% hours observed": {k: 100 * (1 - m.mean()) for k, m in periods.items()},
                             "missing hours": {k: int(m.sum()) for k, m in periods.items()},
                             "longest gap (hours)": {k: gap_table(m)["hours"].max() for k, m in periods.items()}})
MAX_FILL_GAP = 12                                                   # [derived] longest gap in the window (9 h) plus a margin
window_check

,% hours observed,missing hours,longest gap (hours)
before 2016 (10/2012 – 12/2015),61.469,10964,7386
study window (01/2016 – 09/2018),95.800,1012,9


Giai đoạn 01/2016 – 09/2018 có **95.8% số giờ có quan sát, khoảng trống dài nhất chỉ 9 giờ**, đủ ngắn để điền (4.1). Giai đoạn dài 2 năm 9 tháng, nên mỗi mùa có mặt ít nhất hai lần.

### 3.3 Giá trị vô lý và nhãn ngày lễ

Mỗi biến thời tiết dạng số được so với khoảng giá trị có thể xảy ra trong thực tế.

In [12]:
VALID_RANGES = {"temp_c": (-60, 60), "rain_1h": (0, 100)}   # physically possible ranges (°C, mm per hour)
display(pd.DataFrame([{"column": c, "valid range": f"[{lo}, {hi}]", "below": int((regular[c] < lo).sum()),
                       "above": int((regular[c] > hi).sum()), "observed min": regular[c].min(),
                       "observed max": regular[c].max()} for c, (lo, hi) in VALID_RANGES.items()]))
holiday_rows = raw[raw["holiday"] != "None"]
print("Hour of day of the rows carrying a holiday name:", holiday_rows["timestamp"].dt.hour.value_counts().to_dict())

,column,valid range,below,above,observed min,observed max
0,temp_c,"[-60, 60]",10,0,-273.150,36.920
1,rain_1h,"[0, 100]",0,1,0.000,"9,831.300"


Hour of day of the rows carrying a holiday name: {0: 61}


In [13]:
# Figure: the two sensor errors before and after the rule above (whole file, daily extremes keep the notebook light)
RAIN_RECORD_1H = 305   # mm, world record for 60 minutes (WMO Weather and Climate Extremes Archive: Holt, Missouri, 1947)
bad_t = regular["temp_c"][regular["temp_c"] < VALID_RANGES["temp_c"][0]]
bad_r = regular["rain_1h"][regular["rain_1h"] > VALID_RANGES["rain_1h"][1]]
temp_ok = regular["temp_c"].where(regular["temp_c"].between(*VALID_RANGES["temp_c"]))
rain_ok = regular["rain_1h"].where(regular["rain_1h"].between(*VALID_RANGES["rain_1h"]))
panels = [(regular["temp_c"].resample("D").min(), regular["rain_1h"].resample("D").max()),
          (temp_ok.resample("D").min(), rain_ok.resample("D").max())]

fig = make_subplots(rows=2, cols=2, vertical_spacing=0.16, horizontal_spacing=0.08,
                    subplot_titles=("Temperature, raw (daily minimum)", "Rain in one hour, raw (daily maximum)",
                                    "Temperature after removing values outside [−60, 60] °C",
                                    "Rain after removing values above 100 mm"))
for row, (t, r) in enumerate(panels, start=1):
    fig.add_trace(go.Scatter(x=t.index, y=t.values, mode="lines", line=dict(color=BLUE, width=1), showlegend=False,
                             hovertemplate="%{x|%d/%m/%Y}: %{y:.1f} °C<extra></extra>"), row=row, col=1)
    fig.add_trace(go.Scatter(x=r.index, y=r.values, mode="lines", line=dict(color=AQUA, width=1), showlegend=False,
                             hovertemplate="%{x|%d/%m/%Y}: %{y:,.1f} mm<extra></extra>"), row=row, col=2)
fig.add_trace(go.Scatter(x=bad_t.index, y=bad_t.values, mode="markers", name=f"{len(bad_t)} hours at 0 K (−273 °C)",
                         marker=dict(color=RED, size=9, line=dict(color="#ffffff", width=1))), row=1, col=1)
fig.add_trace(go.Scatter(x=bad_r.index, y=bad_r.values, mode="markers", name=f"{len(bad_r)} hour at {bad_r.iloc[0]:,.0f} mm",
                         marker=dict(color=RED, size=9, symbol="diamond", line=dict(color="#ffffff", width=1))), row=1, col=2)
fig.add_hline(y=RAIN_RECORD_1H, line_dash="dot", line_color=INK_2, row=1, col=2,
              annotation_text=f"world 1-hour record ≈ {RAIN_RECORD_1H} mm", annotation_position="top left",
              annotation_font_color=INK_2)
for row in (1, 2):
    fig.add_hrect(y0=VALID_RANGES["temp_c"][0], y1=VALID_RANGES["temp_c"][1], fillcolor=AQUA, opacity=0.07,
                  line_width=0, row=row, col=1)
    fig.update_yaxes(title_text="°C", row=row, col=1)
    fig.update_yaxes(title_text="mm", row=row, col=2)
show_fig(fig, f"Sensor errors: {len(bad_t)} hours at 0 K and one hour of rain {bad_r.iloc[0] / RAIN_RECORD_1H:.0f}× the world record "
              "squeeze the real data; once removed, the seasonal cycle appears", "ts_invalid_values", height=640,
         legend=dict(orientation="h", yanchor="top", y=-0.08, x=0), margin=dict(b=90))
print(f"temperature after the rule: {temp_ok.min():.1f} to {temp_ok.max():.1f} °C "
      f"(5–95%: {temp_ok.quantile(0.05):.0f} to {temp_ok.quantile(0.95):.0f} °C) | "
      f"rain after the rule: max {rain_ok.max():.1f} mm, median of rainy hours {rain_ok[rain_ok > 0].median():.2f} mm | "
      f"9,831 mm = {bad_r.iloc[0] / RAIN_RECORD_1H:.1f}× the record")

temperature after the rule: -29.8 to 36.9 °C (5–95%: -15 to 27 °C) | rain after the rule: max 55.6 mm, median of rainy hours 0.51 mm | 9,831 mm = 32.2× the record


In [14]:
# Figure: the holiday name sits on the 0h row only, while the whole day is affected
LABEL_DAY = pd.Timestamp("2016-07-04")   # Independence Day, a Monday inside the study window with all 24 hours observed
day_hours = pd.date_range(LABEL_DAY, periods=24, freq=FREQ)
label = raw.drop_duplicates("timestamp").set_index("timestamp")["holiday"].reindex(day_hours)
y_day = regular[TARGET].reindex(day_hours)
assert y_day.notna().all() and label.notna().all()
same_weekday = [LABEL_DAY + pd.Timedelta(weeks=k) for k in (-4, -3, -2, -1, 1, 2, 3, 4)]
usual_day = pd.DataFrame({d: regular[TARGET].reindex(pd.date_range(d, periods=24, freq=FREQ)).to_numpy()
                          for d in same_weekday}).median(axis=1)
named = label.ne("None").to_numpy()

fig = go.Figure()
fig.add_trace(go.Bar(x=np.arange(24)[named], y=y_day.to_numpy()[named], marker_color=ORANGE,
                     name=f"holiday = '{label[named].iloc[0]}' ({named.sum()} row)"))
fig.add_trace(go.Bar(x=np.arange(24)[~named], y=y_day.to_numpy()[~named], marker_color=BLUE,
                     name=f"holiday = 'None' ({(~named).sum()} rows)"))
fig.add_trace(go.Scatter(x=np.arange(24), y=usual_day.values, mode="lines+markers", line=dict(color=INK_2, dash="dot"),
                         marker=dict(size=5), name="ordinary Monday (median of the 4 Mondays before and after)"))
day_change = y_day.sum() / usual_day.sum() - 1
show_fig(fig, f"Only the 0h row of Independence Day 2016 carries the holiday name, yet the whole day is "
              f"{-day_change:.0%} below an ordinary Monday", "ts_holiday_label", height=420,
         xaxis_title="Hour of day", yaxis_title="Vehicles/hour", bargap=0.15)
print(f"{LABEL_DAY:%a %d/%m/%Y}: rows named as holiday at hours {list(np.flatnonzero(named))} | "
      f"day total {y_day.sum():,.0f} vs ordinary Monday {usual_day.sum():,.0f} ({day_change:+.0%}) | "
      f"7h–9h: {y_day.iloc[7:10].mean():,.0f} vs {usual_day.iloc[7:10].mean():,.0f}")

Mon 04/07/2016: rows named as holiday at hours [np.int64(0)] | day total 46,678 vs ordinary Monday 79,688 (-41%) | 7h–9h: 1,147 vs 5,224


**10 giờ có nhiệt độ 0 K (−273 °C) và 1 giờ có lượng mưa 9 831 mm.** Hình 2 đặt hai giá trị này cạnh phần còn lại của dữ liệu. Khi còn giữ chúng, trục nhiệt độ bị kéo xuống −273 °C và trục lượng mưa lên gần 10 000 mm, nên toàn bộ dữ liệu thật bị ép thành một đường gần phẳng. Sau khi loại các giá trị ngoài khoảng hợp lệ, nhiệt độ nằm trong khoảng −29.8 đến 36.9 °C và hiện rõ chu kỳ mùa, còn lượng mưa lớn nhất chỉ 55.6 mm trong một giờ (trung vị của các giờ có mưa là 0.51 mm). Con số 9 831 mm gấp 32 lần kỷ lục thế giới về lượng mưa trong 60 phút (khoảng 305 mm, theo kho kỷ lục của WMO), nên đây chắc chắn là lỗi cảm biến. Hai ngưỡng [−60, 60] °C và 100 mm rộng hơn nhiều so với khí hậu Minneapolis, nên chúng chỉ bắt đúng 11 giờ lỗi.

**Tên ngày lễ chỉ được ghi ở dòng 0h** (61 dòng), 23 giờ còn lại của ngày đó vẫn ghi `None`. Hình 3 lấy ví dụ Quốc khánh 4/7/2016: chỉ cột 0h mang tên ngày lễ, nhưng cả ngày thấp hơn một thứ Hai bình thường 41%, và đỉnh sáng 7h–9h chỉ còn 1 147 xe so với 5 224 xe. Nếu dùng nguyên cột này, mô hình chỉ biết giờ nửa đêm là ngày lễ, nên tên ngày lễ phải được lan cho cả ngày (4.1).

### 3.4 Tóm tắt các vấn đề và đề xuất xử lý

Dữ liệu thô **chưa phải một chuỗi thời gian đều**: có dòng trùng, có khoảng trống, có lỗi đo và nhãn ngày lễ chỉ ghi ở một dòng. Bảng sau tóm tắt các bước làm sạch sẽ thực hiện ở mục 4.

| Vấn đề | Quy mô | Đề xuất | Lý do |
|---|---|---|---|
| Dòng trùng mốc giờ | 7 629 dòng (15.8%) | gộp về một dòng mỗi giờ, giữ `n_weather` | số xe trong các dòng trùng luôn bằng nhau |
| Khoảng trống dài | 22.8% số giờ, dài nhất 7 386 giờ | chỉ giữ 01/2016 – 09/2018 | không thể điền mười tháng |
| Khoảng trống ngắn trong giai đoạn giữ lại | 4.2% số giờ, dài nhất 9 giờ | điền từ quá khứ, gắn cờ, không dùng làm nhãn | xem 4.1 |
| Lỗi cảm biến | 10 giờ 0 K, 1 giờ 9 831 mm | đổi thành giá trị thiếu rồi điền như trên | giá trị không thể xảy ra |
| Tên ngày lễ chỉ ở dòng 0h | 61 dòng | lan cho cả ngày | ngày lễ là thuộc tính của cả ngày |

Phân phối của nhãn và các đầu vào được xem ở mục 5, trên tập train đã làm sạch. Muốn biết một giờ có bất thường không thì phải biết mức thường lệ của giờ đó, nên việc tìm ngoại lai của nhãn được để đến khi có chuỗi sạch (5.4).

<a id="s4"></a>
## 4. Tiền xử lý

### 4.1 Làm sạch chuỗi

Hàm `clean_series` làm lần lượt các bước ở 3.4: gộp dòng trùng và dựng lại lịch đầy đủ, cắt về giai đoạn nghiên cứu, đổi giá trị vô lý thành giá trị thiếu, điền các khoảng trống ngắn, rồi lan tên ngày lễ cho cả ngày.

Khoảng trống được điền bằng **forward fill** (lấy giá trị quan sát gần nhất trước đó), không dùng nội suy tuyến tính. Lý do nằm ở ràng buộc của mục 2. Nội suy cần cả quan sát nằm *sau* khoảng trống: nếu thiếu $y_{t-1}$, giá trị nội suy là trung bình của $y_{t-2}$ và $y_t$, tức là một đầu vào của dòng $t$ đã chứa một phần nhãn $y_t$. Forward fill thì chỉ nhìn về quá khứ. Các giờ được điền mang cờ `is_filled` và **chỉ dùng làm đầu vào**; mô hình chỉ học và chỉ được chấm trên số xe thật.

In [15]:
holiday_by_day = holiday_rows.set_index(holiday_rows["timestamp"].dt.normalize())["holiday"]
holiday_by_day = holiday_by_day[~holiday_by_day.index.duplicated()]


def holiday_of(index):
    """Holiday name ("None" otherwise) for every timestamp of `index`, spread over the whole day."""
    return pd.Series(index.normalize().map(holiday_by_day), index=index).fillna("None")


def short_gap_mask(s, max_len):
    """True on NaN cells that belong to a run of at most max_len consecutive NaN."""
    isna = s.isna()
    run_len = isna.groupby(isna.ne(isna.shift()).cumsum()).transform("sum")
    return isna & (run_len <= max_len)


def clean_series(df):
    out = to_regular(df).loc[WINDOW_START:WINDOW_END]
    for c, (low, high) in VALID_RANGES.items():
        out.loc[(out[c] < low) | (out[c] > high), c] = np.nan          # sensor errors -> missing
    out["is_filled"] = out[TARGET].isna().astype(int)
    for c in [TARGET, *NUM_EXOG, *CAT_EXOG]:
        out[c] = out[c].where(~short_gap_mask(out[c], MAX_FILL_GAP), out[c].ffill())   # past values only
    out["holiday"] = holiday_of(out.index)
    return out


ts = clean_series(raw)
assert ts[[TARGET, *NUM_EXOG, *CAT_EXOG]].notna().all().all(), "a gap longer than MAX_FILL_GAP is left"
pd.Series({"raw rows": len(raw), "unique hours in the file": raw["timestamp"].nunique(),
           "hours in the study window": len(ts), "observed hours (targets)": int((ts["is_filled"] == 0).sum()),
           "filled hours (inputs only)": int(ts["is_filled"].sum()),
           "hours labelled as holiday": int(ts["holiday"].ne("None").sum())}, name="count").to_frame()

,count
raw rows,48204
unique hours in the file,40575
hours in the study window,24096
observed hours (targets),23084
filled hours (inputs only),1012
hours labelled as holiday,672


Chuỗi sạch `ts` có **24 096 giờ liên tục, mỗi giờ một dòng, không còn giá trị thiếu**. Trong đó 1 012 giờ (4.2%) là giá trị điền từ quá khứ và chỉ dùng làm đầu vào.

### 4.2 Chia train / validation / test theo thời gian

Dữ liệu được chia theo thời gian, không xáo trộn. Nếu chia ngẫu nhiên thì giờ ngay trước một giờ test có thể nằm trong train, và mô hình được chấm trên một câu hỏi gần như đã biết đáp án. Phần để riêng từ đầu mục 3 được tách đôi: **validation** (12/2017 – 04/2018) dùng để so sánh và chọn mô hình, **test** (05/2018 – 09/2018) chỉ chấm một lần. **Train** là phần còn lại của giai đoạn nghiên cứu (01/2016 – 11/2017). Việc chia được làm trước phân tích khám phá, để mục 5 chỉ nhìn thấy train.

In [16]:
TRAIN_END = HOLDOUT_START - pd.Timedelta(hours=1)
VAL_END = pd.Timestamp("2018-04-30 23:00")
SPLITS = {"train": ts.loc[:TRAIN_END], "validation": ts.loc[HOLDOUT_START:VAL_END], "test": ts.loc[VAL_END:].iloc[1:]}
display(pd.DataFrame({
    "from": {k: p.index.min() for k, p in SPLITS.items()}, "to": {k: p.index.max() for k, p in SPLITS.items()},
    "hours": {k: len(p) for k, p in SPLITS.items()},
    f"mean {TARGET}": {k: p.loc[p["is_filled"] == 0, TARGET].mean() for k, p in SPLITS.items()},
    "filled hours": {k: int(p["is_filled"].sum()) for k, p in SPLITS.items()},
    "% hours with snow": {k: (p["weather_main"] == "Snow").mean() * 100 for k, p in SPLITS.items()}}))

fig = go.Figure()
for (name, part), color in zip(SPLITS.items(), [BLUE, ORANGE, AQUA]):
    daily = part[TARGET].resample("D").mean()
    fig.add_trace(go.Scatter(x=daily.index, y=daily.values, mode="lines", name=f"{name} ({len(part):,} hours)",
                             line=dict(color=color, width=1.6)))
for cut in (HOLDOUT_START, VAL_END):
    fig.add_vline(x=cut, line_dash="dot", line_color=INK_2)
show_fig(fig, "Chronological split: train → validation (winter, 18% snowy hours) → test (summer, no snow)",
         "ts_split", height=400, yaxis_title="Vehicles/hour (daily mean)", hovermode="x unified")

,from,to,hours,mean traffic_volume,filled hours,% hours with snow
train,2016-01-01,2017-11-30 23:00:00,16800,"3,297.686",989,5.726
validation,2017-12-01,2018-04-30 23:00:00,3624,"3,247.507",17,17.632
test,2018-05-01,2018-09-30 23:00:00,3672,"3,358.968",6,0.000


Mức trung bình của ba tập gần nhau (3 248 – 3 359 xe/giờ), nhưng **mùa thì khác**: 17.6% số giờ của validation có tuyết, còn test là mùa hè, không có giờ nào có tuyết (Hình 4). Do đó điểm trên test không nói được mô hình xử lý bão tuyết tốt đến đâu; phần mùa đông chỉ được đo trên validation.

<a id="s5"></a>
## 5. Phân tích khám phá trên chuỗi đã làm sạch

Mục này tìm **hình dạng cuối cùng** của dữ liệu: xu hướng, chu kỳ, phần còn lại sau chu kỳ, ngoại lai, mức phụ thuộc vào quá khứ, tác động của thời tiết và ngày lễ. Mọi phân tích **chỉ dùng tập train**. Thống kê mô tả chỉ tính trên giờ có quan sát thật; phân rã chu kỳ và tự tương quan cần chuỗi liền mạch nên dùng chuỗi đã điền. Cuối mỗi mục có một dòng **gợi ý** cho bước mô hình hóa; còn quyết định cụ thể được đưa ra ở mục 6 và 7.

In [17]:
eda = SPLITS["train"]
y_eda = eda[TARGET].where(eda["is_filled"] == 0)    # observed hours only (NaN on filled hours)
y_cont = eda[TARGET]                                 # continuous series (short gaps filled): for MSTL and ACF only
hol_name = eda["holiday"]
day_total = y_cont.resample("D").sum()
is_hol_day = hol_name.resample("D").first().ne("None")
how = pd.Series(eda.index.dayofweek * 24 + eda.index.hour, index=eda.index)   # hour of the week: 0 = Mon 0h
print(f"EDA period {eda.index.min():%d/%m/%Y} – {eda.index.max():%d/%m/%Y}: {len(eda):,} hours, "
      f"{int(y_eda.notna().sum()):,} observed, {int(is_hol_day.sum())} holiday days")

EDA period 01/01/2016 – 30/11/2017: 16,800 hours, 15,811 observed, 20 holiday days


### 5.1 Phân phối của nhãn, xu hướng dài hạn và phân phối có điều kiện theo giờ

In [18]:
y_obs = y_eda.dropna()   # observed hours of the training period (filled hours excluded)
# colour each hour by its clock time, to see which hours make up each peak
TIME_GROUPS = {"night (22h–4h)": [22, 23, 0, 1, 2, 3, 4],
               "transition (5h–6h, 19h–21h)": [5, 6, 19, 20, 21],
               "daytime (7h–18h)": list(range(7, 19))}
hour_of = y_obs.index.hour
bins = np.arange(0, 7600, 200)
fig = go.Figure()
for (name, hrs), color in zip(TIME_GROUPS.items(), [MUTED, YELLOW, BLUE]):
    fig.add_trace(hist_bar(y_obs[np.isin(hour_of, hrs)], bins=bins, color=color))
    fig.data[-1].update(name=name, showlegend=True)
fig.add_vline(x=y_obs.median(), line_dash="dot", line_color=INK_2, annotation_text=f"median {y_obs.median():,.0f}",
              annotation_position="top left", annotation_font_color=INK_2)
is_night, is_day = np.isin(hour_of, TIME_GROUPS["night (22h–4h)"]), np.isin(hour_of, TIME_GROUPS["daytime (7h–18h)"])
show_fig(fig, f"{TARGET} is bimodal because night and daytime hours are mixed: "
              f"{(y_obs[is_night] < 1000).mean():.0%} of night hours are below 1,000, "
              f"{(y_obs[is_day] >= 4000).mean():.0%} of daytime hours are at 4,000 or more", "ts_target_hist",
         xaxis_title="Vehicles/hour", yaxis_title="Hours", barmode="stack")
low, high = y_obs < 1000, y_obs >= 4000
print(f"{len(y_obs):,} observed hours | range {y_obs.min():,.0f}–{y_obs.max():,.0f} | mean {y_obs.mean():,.0f}, "
      f"median {y_obs.median():,.0f}, std {y_obs.std():,.0f} | < 1,000: {low.mean():.1%}, "
      f">= 4,000: {high.mean():.1%}, zeros: {int((y_obs == 0).sum())} | "
      f"skewness {y_obs.skew():.2f}, excess kurtosis {y_obs.kurt():.2f}")
print(f"night hours below 1,000: {(y_obs[is_night] < 1000).mean():.1%} | daytime hours at 4,000 or more: {(y_obs[is_day] >= 4000).mean():.1%}")
print(f"hours below 1,000 that are night hours: {is_night[low.to_numpy()].mean():.1%} | "
      f"hours at 4,000 or more that are daytime hours: {is_day[high.to_numpy()].mean():.1%}")

15,811 observed hours | range 0–7,280 | mean 3,298, median 3,487, std 1,970 | < 1,000: 21.9%, >= 4,000: 44.9%, zeros: 2 | skewness -0.13, excess kurtosis -1.29
night hours below 1,000: 67.6% | daytime hours at 4,000 or more: 84.0%
hours below 1,000 that are night hours: 91.4% | hours at 4,000 or more that are daytime hours: 92.8%


Trên 15 811 giờ có quan sát của train, nhãn nằm trong khoảng 0 đến 7 280 xe/giờ, trung vị 3 487. Phân phối **có hai đỉnh**, một quanh 300 xe/giờ và một quanh 4 700 xe/giờ (Hình 5). Để xem mỗi đỉnh gồm những giờ nào, Hình 5 tô màu từng giờ theo thời điểm trong ngày: ban đêm (22h–4h), ban ngày (7h–18h) và giờ chuyển tiếp ở giữa (5h–6h, 19h–21h).

Đỉnh trái gần như toàn giờ đêm, vì **91.4%** số giờ dưới 1 000 xe là giờ đêm. Ngược lại, **92.8%** số giờ từ 4 000 xe trở lên là giờ ban ngày. Còn giờ chuyển tiếp tạo thành một bướu nhỏ quanh 2 800 xe. Như vậy, hai đỉnh đến từ việc trộn giờ đêm với giờ ngày. Ngoài ra, phân phối gần đối xứng (skewness −0.13), nên lấy log không giúp gì, và chúng tôi giữ nhãn ở thang gốc.

Trước khi xem kỹ từng giờ, chúng tôi kiểm tra lưu lượng có xu hướng dài hạn hay không.

In [19]:
daily = y_eda.resample("D").mean()
fig = go.Figure()
fig.add_trace(go.Scatter(x=daily.index, y=daily.values, mode="lines", line=dict(color=BLUE, width=1), opacity=0.7,
                         name="daily mean", hovertemplate="%{x|%a %d/%m/%Y}: %{y:,.0f}<extra></extra>"))
fig.add_trace(go.Scatter(x=daily.index, y=daily.rolling(28, center=True, min_periods=10).mean().values, mode="lines",
                         line=dict(color=ORANGE, width=3), name="28-day rolling mean"))
show_fig(fig, "Daily mean traffic on the training period: no long-term trend, short dips around holidays and new year",
         "ts_trend_daily", height=400, yaxis_title="Vehicles/hour (daily mean)", hovermode="x unified")
print(f"28-day rolling mean stays between {daily.rolling(28, center=True).mean().min():,.0f} and "
      f"{daily.rolling(28, center=True).mean().max():,.0f} vehicles/hour; overall mean {y_eda.mean():,.0f}")

28-day rolling mean stays between 2,850 and 3,523 vehicles/hour; overall mean 3,298


Trung bình trượt 28 ngày dao động trong khoảng 2 850 – 3 520 xe/giờ suốt gần hai năm, không đi lên hay đi xuống, chỉ chùng xuống một thời gian ngắn quanh kỳ nghỉ cuối năm (Hình 6). **Chuỗi không có xu hướng dài hạn.**

Hình 7 kiểm tra nhận định về hai đỉnh của histogram ở trên. Nếu hai đỉnh của histogram là do trộn nhiều giờ có mức khác nhau, thì khi xét riêng từng giờ trong ngày, phân phối phải còn một đỉnh và hẹp lại.

In [20]:
is_wkend = np.where(eda.index.dayofweek >= 5, "Weekend", "Weekday (Mon–Fri)")
qs = y_eda.groupby([is_wkend, eda.index.hour]).quantile([0.05, 0.25, 0.5, 0.75, 0.95]).unstack()
hours = np.arange(24)
fig = make_subplots(rows=1, cols=2, shared_yaxes=True, horizontal_spacing=0.04,
                    subplot_titles=("Weekday (Mon–Fri)", "Weekend"))
for c, (name, color, rgb) in enumerate([("Weekday (Mon–Fri)", BLUE, "42,120,214"), ("Weekend", ORANGE, "235,104,52")], start=1):
    q = qs.loc[name]
    band(fig, hours, q[0.05].values, q[0.95].values, f"rgba({rgb},0.15)", row=1, col=c)
    band(fig, hours, q[0.25].values, q[0.75].values, f"rgba({rgb},0.35)", row=1, col=c)
    fig.add_trace(go.Scatter(x=hours, y=q[0.5].values, mode="lines+markers", line=dict(color=color, width=2.5),
                             marker=dict(size=6), showlegend=False,
                             hovertemplate="%{x}h: median %{y:,.0f}<extra>" + name + "</extra>"), row=1, col=c)
fig.update_xaxes(title_text="Hour of day", dtick=3)
fig.update_yaxes(title_text="Vehicles/hour", col=1)
show_fig(fig, "Within one hour of the day the distribution is unimodal and narrow (dark band = 25–75%, light = 5–95%)",
         "ts_target_by_hour", height=440)
wd_med = qs.loc["Weekday (Mon–Fri)", 0.5]
print(f"weekday median: lowest {wd_med.min():,.0f} at {wd_med.idxmin()}h, highest {wd_med.max():,.0f} at {wd_med.idxmax()}h; "
      f"weekend median peak {qs.loc['Weekend', 0.5].max():,.0f} at {qs.loc['Weekend', 0.5].idxmax()}h")

weekday median: lowest 295 at 2h, highest 6,365 at 16h; weekend median peak 4,515 at 13h


Khi xét riêng từng giờ, phân phối **chỉ còn một đỉnh và hẹp**. Ngày thường, trung vị đi từ 295 xe lúc 2h lên 6 365 xe lúc 16h, với một đỉnh phụ lúc 7h (Hình 7). Vậy nhận định ở 5.1 là đúng: hai đỉnh của histogram là do trộn các giờ, không phải hai nhóm riêng. Cuối tuần không chỉ thấp hơn mà **có dạng khác hẳn**: một đỉnh khoảng 4 500 xe lúc 13h, thay cho hai đỉnh giờ đi làm và giờ tan làm. Như vậy tác động của giờ và thứ **không cộng đơn giản với nhau**.

Hình 7 cũng gợi ra hai chu kỳ: **một ngày (24 giờ)** và **một tuần (168 giờ)**, vì ngày thường và cuối tuần khác nhau. Mục 5.2 đo độ mạnh của từng chu kỳ.

*Gợi ý:* mô hình cần biết giờ và thứ, và cần học được cách hai yếu tố này kết hợp với nhau.

In [21]:
SEASONAL_PERIODS = [24, 168]   # [derived] candidate cycles seen in Figure 7; their strength is measured in 5.2

### 5.2 Độ mạnh của chu kỳ ngày và tuần

Chúng tôi tách chuỗi bằng MSTL (Multiple Seasonal-Trend decomposition using LOESS) thành

$$y_t = T_t + S^{(24)}_t + S^{(168)}_t + R_t,$$

gồm xu hướng, mùa vụ ngày, mùa vụ tuần và phần dư. Mỗi thành phần được đánh giá qua tỷ lệ phương sai của nó và độ mạnh mùa vụ $F_S = \max\left(0,\ 1 - \operatorname{Var}(R)/\operatorname{Var}(S + R)\right)$ (Wang, Smith và Hyndman, 2006). Để đối chiếu, chúng tôi tính thêm tỷ lệ phương sai giải thích được khi chỉ biết giờ trong ngày hoặc giờ trong tuần ($\eta^2$).

In [22]:
def eta_squared(y, groups):
    """Share of the variance of y explained by the group means."""
    return 1 - ((y - y.groupby(groups).transform("mean")) ** 2).sum() / ((y - y.mean()) ** 2).sum()


decomposition = MSTL(y_cont, periods=SEASONAL_PERIODS, windows=[p + 1 for p in SEASONAL_PERIODS]).fit()
components = pd.concat([y_cont.rename("observed"), decomposition.trend.rename("trend"), decomposition.seasonal,
                        decomposition.resid.rename("residual")], axis=1)
var_share = (components.drop(columns="observed").var() / components["observed"].var() * 100).rename("% of variance")
strength = pd.Series({c: max(0, 1 - components["residual"].var() / (components["residual"] + components[c]).var())
                      for c in decomposition.seasonal.columns}, name="seasonal strength F_S")
display(pd.concat([var_share, strength], axis=1))
print(f"eta²: hour of day explains {eta_squared(y_eda, eda.index.hour):.1%} of the variance, "
      f"hour of week {eta_squared(y_eda, how):.1%}")

,% of variance,seasonal strength F_S
trend,1.258,NaN
seasonal_24,78.376,0.944
seasonal_168,15.171,0.765
residual,4.659,NaN


eta²: hour of day explains 78.7% of the variance, hour of week 94.2%


In [23]:
recent = components.loc["2017-03-06":"2017-03-26"]   # three full weeks (Monday to Sunday) inside train
fig = make_subplots(rows=len(recent.columns), cols=1, shared_xaxes=True, vertical_spacing=0.035,
                    subplot_titles=[c + (f": {var_share[c]:.0f}% of variance" if c in var_share else "") for c in recent.columns])
for i, (c, color) in enumerate(zip(recent.columns, [BLUE, ORANGE, AQUA, YELLOW, RED]), start=1):
    fig.add_trace(go.Scatter(x=recent.index, y=recent[c], mode="lines", line=dict(color=color, width=1.6), showlegend=False,
                             hovertemplate="%{x|%a %d/%m %H:%M}: %{y:,.0f}<extra>" + c + "</extra>"), row=i, col=1)
show_fig(fig, "MSTL over three weeks of March 2017: observed = trend + daily + weekly + residual", "ts_mstl", height=820)

Chu kỳ ngày chiếm **78% phương sai** ($F_S = 0.94$), chu kỳ tuần **15%** ($F_S = 0.77$), xu hướng 1% và phần dư 5% (bảng trên, Hình 8). Cách tính $\eta^2$ cho kết quả tương tự: biết giờ trong ngày giải thích được 78.7% phương sai, biết **giờ trong tuần giải thích được 94.2%**. Cả hai chu kỳ đều rõ.

*Gợi ý:* chỉ lặp lại cùng giờ tuần trước đã bắt được phần lớn biến động, nên đây là một mốc so sánh khó vượt. Cũng vì lịch giải thích gần hết phương sai, các độ đo kiểu "tỷ lệ phương sai giải thích" sẽ cao với hầu như mọi mô hình. Cái khó thật sự nằm ở khoảng 6% còn lại.

### 5.3 Độ lệch khỏi mức thường lệ

Lấy trung vị của cùng thứ, cùng giờ làm mức thường lệ, độ lệch tương đối của một giờ là $d_t = \left(y_t - \operatorname{median}(y \mid \text{thứ}, \text{giờ})\right) / \operatorname{median}(y \mid \text{thứ}, \text{giờ})$. Hình 9 cho thấy độ phân tán thay đổi theo mức ra sao, và độ lệch có phân phối dạng gì.

In [24]:
by_how = y_eda.groupby(how)
cells = pd.DataFrame({"mean": by_how.mean(), "std": by_how.std(), "weekend": np.arange(168) >= 120})
quiet, busy = cells["mean"] < 1000, cells["mean"] >= 4000
usual = by_how.transform("median")
dev = y_eda - usual
rel_dev = dev / usual

fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.1, column_widths=[0.42, 0.58],
                    subplot_titles=("Spread grows with the level (dot = one hour of the week)",
                                    "Deviation from the usual value (log scale)"))
for wk, name, color in [(False, "Mon–Fri", BLUE), (True, "Sat–Sun", ORANGE)]:
    part = cells[cells["weekend"] == wk]
    fig.add_trace(go.Scatter(x=part["mean"], y=part["std"], mode="markers", name=name,
                             marker=dict(color=color, size=7, opacity=0.8)), row=1, col=1)
edges = np.arange(-100, 102.5, 2.5)
counts, _ = np.histogram(np.clip(rel_dev.dropna() * 100, -100, 100), bins=edges)
centers = (edges[:-1] + edges[1:]) / 2
fig.add_trace(go.Bar(x=centers, y=np.where(counts > 0, counts, np.nan), width=2.3, showlegend=False,
                     marker_color=[BLUE if abs(c) <= 10 else (RED if c < -30 else MUTED) for c in centers],
                     hovertemplate="%{x:+.1f}%: %{y:,} hours<extra></extra>"), row=1, col=2)
fig.add_annotation(x=0, y=np.log10(counts.max()) + 0.15, xref="x2", yref="y2", showarrow=False,
                   text=f"{(rel_dev.abs() <= 0.10).mean():.0%} within ±10%", font_color=BLUE)
fig.add_annotation(x=-70, y=np.log10(counts.max()) - 1.2, xref="x2", yref="y2", showarrow=False,
                   text=f"long tail of drops:<br>{(rel_dev < -0.30).mean():.1%} below −30%", font_color=RED)
fig.update_xaxes(title_text="Mean of that hour of the week (vehicles/hour)", row=1, col=1)
fig.update_yaxes(title_text="Std (vehicles/hour)", row=1, col=1)
fig.update_xaxes(title_text="(y − usual) / usual, % (clipped to ±100%)", row=1, col=2)
fig.update_yaxes(title_text="Hours", type="log", dtick=1, row=1, col=2)
show_fig(fig, "What is left after the hour of the week: spread grows with the level, a narrow core and a heavy tail of drops",
         "ts_target_spread", height=460, legend=dict(orientation="v", x=0.02, y=0.98, yanchor="top"))
cv = cells["std"] / cells["mean"]
print(f"corr(mean, std) over the 168 hours of the week = {cells['mean'].corr(cells['std']):.2f}; std {cells.loc[quiet, 'std'].mean():,.0f} "
      f"(quiet hours) vs {cells.loc[busy, 'std'].mean():,.0f} (busy hours); coefficient of variation {cv[quiet].mean():.2f} vs {cv[busy].mean():.2f}")
print(f"deviation (vehicles): skewness {dev.skew():.1f}, excess kurtosis {dev.kurt():.0f}, "
      f"1st percentile {dev.quantile(0.01):,.0f}, 99th percentile {dev.quantile(0.99):,.0f}")

corr(mean, std) over the 168 hours of the week = 0.69; std 127 (quiet hours) vs 545 (busy hours); coefficient of variation 0.23 vs 0.11
deviation (vehicles): skewness -3.5, excess kurtosis 28, 1st percentile -1,915, 99th percentile 934


Độ lệch khỏi mức thường lệ có ba đặc điểm (Hình 9):

- **Độ phân tán tăng theo mức.** Tương quan giữa trung bình và độ lệch chuẩn của 168 giờ trong tuần là 0.69; độ lệch chuẩn khoảng 127 xe ở giờ vắng và 545 xe ở giờ đông. Nhưng **tính theo tỷ lệ thì giờ vắng lại nhiễu hơn**: hệ số biến thiên 0.23 so với 0.11.
- **Phần lớn các giờ nằm sát mức thường lệ**: 65% số giờ lệch không quá ±10%.
- **Đuôi trái dày** (excess kurtosis 28, skewness −3.5). Các lần giảm đột ngột lớn hơn nhiều so với các lần tăng: phân vị 1% của độ lệch là −1 915 xe, còn phân vị 99% chỉ là +934 xe.

*Gợi ý:* sai số tính theo phần trăm sẽ bị giờ vắng đẩy lên, còn sai số bình phương sẽ bị vài giờ ở đuôi chi phối; cách đo chất lượng cần tính đến cả hai (7.1). Muốn biết hôm nay có gì khác thường thì chỉ có thể dựa vào giá trị của giờ vừa qua.

### 5.4 Ngoại lai của nhãn

Như đã thấy ở 5.3, z-score tính trên toàn chuỗi không dùng được, vì giá trị ban đêm đặt cạnh giá trị ban ngày trông như ngoại lai. Chúng tôi trừ đi hồ sơ mùa vụ (trung vị của cùng giờ trong tuần) để có phần dư $r_t$, rồi chuẩn hóa phần dư trong cửa sổ trượt 168 giờ đặt giữa, theo hai cách:

$$z^{\text{std}}_t = \frac{r_t - \operatorname{mean}_{168}(r)}{\operatorname{std}_{168}(r)}, \qquad z^{\text{robust}}_t = \frac{r_t - \operatorname{median}_{168}(r)}{1.4826 \cdot \operatorname{MAD}_{168}(r)}.$$

Một giờ bị đánh dấu khi $|z_t| > 3.5$ (ngưỡng Iglewicz–Hoaglin). Để có thứ đối chiếu, chúng tôi gọi **giờ đóng đường** là giờ ban ngày (7h–19h) có dưới 300 xe trong khi bình thường giờ đó có trên 3 000 xe.

In [25]:
Z_THRESHOLD = 3.5   # Iglewicz-Hoaglin cut-off for robust z-scores


def robust_outliers(y):
    """Classic and robust z-scores of y against the usual value of the same hour of the week."""
    period = max(SEASONAL_PERIODS)
    position = pd.Series(np.arange(len(y)) % period, index=y.index)
    profile = y.groupby(position).transform("median")
    resid = y - profile
    window = resid.rolling(period, center=True, min_periods=period // 2)
    mad = window.apply(lambda a: np.nanmedian(np.abs(a - np.nanmedian(a))), raw=True)
    z_robust = (resid - window.median()) / (1.4826 * mad)
    z_std = (resid - window.mean()) / window.std()
    return pd.DataFrame({"profile": profile, "z_robust": z_robust, "z_std": z_std, "is_outlier": z_robust.abs() > Z_THRESHOLD})


out_eda = robust_outliers(y_eda)
flag_mad, flag_std = out_eda["is_outlier"], out_eda["z_std"].abs() > Z_THRESHOLD
closure = (y_eda < 300) & (out_eda["profile"] > 3000) & eda.index.hour.isin(range(7, 20))
pd.DataFrame({"flagged hours": [int(flag_std.sum()), int(flag_mad.sum())],
              "% of hours": [flag_std.mean() * 100, flag_mad.mean() * 100],
              f"closure hours found (of {int(closure.sum())})": [int((flag_std & closure).sum()), int((flag_mad & closure).sum())],
              "flagged hours on holidays": [int((flag_std & hol_name.ne("None")).sum()), int((flag_mad & hol_name.ne("None")).sum())]},
             index=["rolling mean / std", "rolling median / MAD"])

,flagged hours,% of hours,closure hours found (of 14),flagged hours on holidays
rolling mean / std,168,1.000,0,32
rolling median / MAD,640,3.810,14,114


In [26]:
seg = slice("2016-07-04", "2016-07-31")
closure_seg = closure[seg]
runs = closure_seg.ne(closure_seg.shift()).cumsum()[closure_seg]          # consecutive closure hours -> one shaded band
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08, row_heights=[0.62, 0.38],
                    subplot_titles=("Traffic vs. the usual value of the same hour of the week",
                                    "Outlier score of each hour: red = median/MAD, orange dotted = mean/std (flagged when |z| > 3.5)"))
fig.add_trace(go.Scatter(x=out_eda["profile"][seg].index, y=out_eda["profile"][seg].values, mode="lines",
                         name="usual (median of the same hour of the week)", line=dict(color=MUTED, width=1.5, dash="dot")), row=1, col=1)
fig.add_trace(go.Scatter(x=y_eda[seg].index, y=y_eda[seg].values, mode="lines", name="actual",
                         line=dict(color=BLUE, width=1.8)), row=1, col=1)
for name, flag, color, symbol in [("median/MAD", flag_mad, RED, "circle"), ("mean/std", flag_std, ORANGE, "x")]:
    pts = y_eda[seg][flag[seg]]
    caught = int((flag & closure)[seg].sum())
    fig.add_trace(go.Scatter(x=pts.index, y=pts.values, mode="markers",
                             name=f"flagged by {name}: catches {caught} of {int(closure_seg.sum())} closure hours",
                             marker=dict(color=color, size=10, symbol=symbol, line=dict(color="#ffffff", width=1.5))), row=1, col=1)
for col, color, dash in [("z_robust", RED, None), ("z_std", ORANGE, "dot")]:
    fig.add_trace(go.Scatter(x=out_eda.loc[seg].index, y=out_eda.loc[seg, col].clip(-15, 15).values, mode="lines",
                             showlegend=False, line=dict(color=color, width=1.6, dash=dash)), row=2, col=1)
for _, hours in closure_seg[closure_seg].groupby(runs):
    for row in (1, 2):
        fig.add_vrect(x0=hours.index.min() - pd.Timedelta(minutes=30), x1=hours.index.max() + pd.Timedelta(minutes=30),
                      fillcolor=YELLOW, opacity=0.3, line_width=0, layer="below", row=row, col=1)
for s in (Z_THRESHOLD, -Z_THRESHOLD):
    fig.add_hline(y=s, line_dash="dot", line_color=INK_2, row=2, col=1)
fig.add_trace(go.Scatter(x=[y_eda[seg].index[0]], y=[None], mode="markers", name=f"road closure ({int(closure_seg.sum())} hours, shaded)",
                         marker=dict(color=YELLOW, size=12, symbol="square", opacity=0.6)), row=1, col=1)
fig.update_xaxes(type="date")
fig.update_yaxes(title_text="Vehicles/hour", row=1, col=1)
fig.update_yaxes(title_text="z (clipped ±15)", row=2, col=1)
show_fig(fig, f"July 2016 road closure: median/MAD flags {int((flag_mad & closure).sum())} of {int(closure.sum())} closure hours, "
              f"mean/std flags {int((flag_std & closure).sum())}", "ts_outlier_compare",
         height=700, hovermode="x unified", margin=dict(t=120), legend=dict(y=1.13))

Median/MAD bắt được **cả 14 giờ đóng đường** của tháng 7/2016, còn mean/std **không bắt được giờ nào** (bảng trên, Hình 10). Lý do là khi đường đóng nhiều giờ liền, chính những giờ đó làm độ lệch chuẩn của cửa sổ tăng vọt, nên z-score của chúng bị kéo nhỏ lại và chúng tự che lẫn nhau (*masking*). Đây là hệ quả của đuôi dày ở 5.3.

Câu hỏi tiếp theo là các giờ bị đánh dấu là lỗi đo hay sự kiện thật. Ô dưới đây xếp các ngày theo số giờ bị đánh dấu.

In [27]:
# Which days carry the most flagged hours, and what happened on them?
day_of = eda.index.normalize()
snow_day = (eda["weather_main"] == "Snow").groupby(day_of).transform("any").to_numpy()
cause = pd.Series(np.select([hol_name.ne("None").to_numpy(), closure.to_numpy(), snow_day],
                            ["holiday", "road closure (7.2 rule)", "day with snow"], "other"), index=eda.index)
print("share of the", int(flag_mad.sum()), "flagged hours by kind of day:",
      (cause[flag_mad].value_counts(normalize=True) * 100).round(1).to_dict())
by_day = pd.DataFrame({"flagged hours": flag_mad.groupby(day_of).sum(),
                       "holiday": hol_name.groupby(day_of).first(),
                       "snow hours": (eda["weather_main"] == "Snow").groupby(day_of).sum(),
                       "day vs usual": y_eda.groupby(day_of).sum() / out_eda["profile"].where(y_eda.notna()).groupby(day_of).sum() - 1})
top_days = by_day.nlargest(10, "flagged hours")
top_days.index = top_days.index.strftime("%a %d/%m/%Y")
top_days

share of the 640 flagged hours by kind of day: {'other': 57.2, 'day with snow': 22.8, 'holiday': 17.8, 'road closure (7.2 rule)': 2.2}


,flagged hours,holiday,snow hours,day vs usual
timestamp,,,,
Mon 29/05/2017,15,Memorial Day,0,-0.364
Sat 23/07/2016,14,None,0,-0.904
Thu 23/11/2017,14,Thanksgiving Day,0,-0.450
Tue 04/07/2017,13,Independence Day,0,-0.407
Thu 24/11/2016,11,Thanksgiving Day,4,-0.460
Mon 30/05/2016,9,Memorial Day,0,-0.347
Sun 24/07/2016,9,None,0,-0.629
Sun 11/12/2016,9,None,24,-0.259
Mon 04/09/2017,9,Labor Day,0,-0.303


**Những ngày có nhiều giờ bị đánh dấu nhất đều có nguyên nhân rõ ràng.** Trong 10 ngày đứng đầu, 7 ngày là ngày lễ, 2 ngày là đợt đóng đường cuối tuần 23–24/7/2016 (cả ngày 23/7 thấp hơn mức thường lệ 90%), và 1 ngày là 11/12/2016 với 24 giờ tuyết. Tuy nhiên, xét theo giờ thì bức tranh khác: chỉ 43% trong 640 giờ bị đánh dấu rơi vào ngày lễ (17.8%), ngày có tuyết (22.8%) hoặc giờ đóng đường (2.2%). 57% còn lại là những giờ lẻ chưa rõ nguyên nhân, nhiều khả năng là sự cố giao thông mà dữ liệu không ghi lại.

Các giờ này vẫn nằm trong khoảng giá trị hợp lý và không có dấu hiệu lỗi đo như ở 3.3. Vì vậy chúng tôi **giữ nguyên** nhãn; nếu "làm sạch" chúng trong test thì điểm số sẽ đẹp giả tạo. Cờ ngoại lai (median/MAD) chỉ dùng khi **phân tích lỗi** ở mục 8, không đưa vào làm đặc trưng, vì nó được tính bằng cửa sổ đặt giữa và do đó có nhìn vào tương lai.

In [28]:
is_outlier = robust_outliers(ts[TARGET].where(ts["is_filled"] == 0))["is_outlier"]   # whole window, for error analysis only
print(f"robust outliers in the study window: {int(is_outlier.sum()):,} hours ({is_outlier.mean():.1%})")

robust outliers in the study window: 1,016 hours (4.2%)


### 5.5 Mức phụ thuộc vào quá khứ: tự tương quan và tính dừng

Ở mục 2 chúng tôi giả định lưu lượng một giờ phụ thuộc vào các giờ trước. Hàm tự tương quan $\text{ACF}(k) = \operatorname{corr}(y_t, y_{t-k})$ dùng để kiểm tra giả định này và cho biết giá trị quá khứ nào giống hiện tại nhất. PACF cho biết lag $k$ còn thêm được gì khi đã biết các lag nhỏ hơn.

In [29]:
N_ACF, N_PACF = 3 * max(SEASONAL_PERIODS), 3 * min(SEASONAL_PERIODS)
acf_vals = acf(y_cont, nlags=N_ACF, fft=True)
pacf_vals = pacf(y_cont, nlags=N_PACF, method="ywm")
ci = 1.96 / np.sqrt(len(y_cont))
lags_axis = np.arange(N_ACF + 1)
fig = make_subplots(rows=2, cols=1, vertical_spacing=0.14,
                    subplot_titles=(f"ACF up to lag {N_ACF}: orange = multiples of 24h, red = multiples of 168h", f"PACF up to lag {N_PACF}"))
fig.add_trace(go.Bar(x=lags_axis, y=acf_vals, marker_line_width=0,
                     marker_color=np.where(lags_axis % 168 == 0, RED, np.where(lags_axis % 24 == 0, ORANGE, BLUE))), row=1, col=1)
fig.add_trace(go.Bar(x=np.arange(N_PACF + 1), y=pacf_vals, marker_color=np.where(np.abs(pacf_vals) > ci, ORANGE, MUTED)), row=2, col=1)
for lag in (12, 24, 168, 336):
    fig.add_annotation(x=lag, y=acf_vals[lag], text=f"lag {lag}: {acf_vals[lag]:.2f}", showarrow=True, arrowcolor=MUTED,
                       ay=30 if acf_vals[lag] < 0 else -30, font_color=INK_2, row=1, col=1)
fig.update_xaxes(title_text="Lag (hours)", row=2, col=1)
fig.update_xaxes(tickvals=np.arange(0, N_ACF + 1, 24), row=1, col=1)
show_fig(fig, "Autocorrelation peaks every 24h and is highest at 168h; PACF is large only at lags 1 and 2",
         "ts_acf_pacf", height=640, showlegend=False, bargap=0.25)
pd.DataFrame({"ACF": [acf_vals[k] for k in [1, 2, 3, 12, 24, 168]],
              "PACF": [pacf_vals[k] if k <= N_PACF else np.nan for k in [1, 2, 3, 12, 24, 168]]},
             index=pd.Index([1, 2, 3, 12, 24, 168], name="lag")).T

lag,1,2,3,12,24,168
ACF,0.905,0.717,0.504,-0.605,0.859,0.934
PACF,0.905,-0.569,0.004,0.101,-0.021,NaN


In [30]:
def stationarity(series, name):
    adf_p = adfuller(series, maxlag=48, autolag="AIC")[1]
    kpss_p = kpss(series, regression="c", nlags="auto")[1]
    return {"series": name, "ADF p (H0: unit root)": adf_p, "KPSS p (H0: stationary)": kpss_p}


pd.DataFrame([stationarity(y_cont, TARGET),
              stationarity(components["trend"] + components["residual"], "deseasonalised (trend + residual)"),
              stationarity(y_cont.diff(168).dropna(), "weekly difference y(t) − y(t−168)")]).set_index("series")

,ADF p (H0: unit root),KPSS p (H0: stationary)
series,,
traffic_volume,0.000,0.010
deseasonalised (trend + residual),0.000,0.010
weekly difference y(t) − y(t−168),0.000,0.100


Kết quả ủng hộ giả định một cách rõ ràng. ACF dao động theo chu kỳ 24 giờ: âm ở lag 12 (−0.61, ngày so với đêm), cao ở lag 24 (0.86) và **cao nhất ở lag 168 (0.93)**, kế đến là lag 1 (0.91) (Hình 11). "Cùng giờ tuần trước" giống hiện tại hơn "cùng giờ hôm qua", vì hôm qua có thể là cuối tuần. PACF lớn ở lag 1 (0.91) và lag 2 (−0.57), **từ lag 3 thì gần bằng 0** (0.004), sau đó chỉ còn vài cụm nhỏ quanh các bội số của 24.

Về tính dừng, ADF bác bỏ giả thuyết có nghiệm đơn vị ở cả ba chuỗi ($p < 0.001$). KPSS bác bỏ tính dừng của chuỗi gốc và chuỗi đã khử mùa vụ ($p = 0.01$), vì mức trung bình trôi chậm theo mùa và vì KPSS rất nhạy khi $n = 16\,800$. Chuỗi sai phân tuần thì qua được cả hai kiểm định. Thêm việc chuỗi không có xu hướng (5.1), chúng tôi kết luận không cần lấy sai phân thường.

*Gợi ý:* các giá trị quá khứ đáng dùng là 1 và 2 giờ trước (theo PACF), cùng giờ hôm qua và cùng giờ tuần trước (theo ACF). Nếu dùng mô hình tự hồi quy, PACF gợi ý bậc khoảng 2; bậc cụ thể sẽ chọn ở 7.3.

In [31]:
LAGS = [1, 2, 24, 168]   # [derived] PACF: lags 1-2; ACF: same hour yesterday and last week

### 5.6 Tác động của thời tiết

Trước khi đo tác động, chúng tôi xem các biến thời tiết trên train trông như thế nào.

In [32]:
w = eda[eda["is_filled"] == 0]   # observed hours of the training period; sensor errors were already removed in 4.1
rain_bins, rain_labels = [-0.001, 0, 0.5, 1, 2, 5, 10, np.inf], ["0", "0–0.5", "0.5–1", "1–2", "2–5", "5–10", "> 10"]
rain_share = pd.cut(w["rain_1h"], rain_bins, labels=rain_labels).value_counts(normalize=True).reindex(rain_labels) * 100
snow_counts = pd.Series({"weather_main = Snow": int((w["weather_main"] == "Snow").sum()),
                         "snow_1h > 0": int((w["snow_1h"] > 0).sum())})
cloud_share = w["clouds_all"].value_counts(normalize=True).sort_index() * 100
wm_share = w["weather_main"].value_counts(normalize=True).sort_values() * 100
wd_counts = w["weather_description"].value_counts()

fig = make_subplots(rows=2, cols=3, horizontal_spacing=0.08, vertical_spacing=0.2,
                    subplot_titles=("temp_c: two seasonal modes", f"rain_1h: {(w['rain_1h'] == 0).mean():.0%} of hours are exactly 0",
                                    "snow_1h almost never records snow", "clouds_all: a few discrete values",
                                    f"weather_main: two groups cover {w['weather_main'].value_counts(normalize=True).head(2).sum():.0%}", "weather_description: long tail of rare values"))
fig.add_trace(hist_bar(w["temp_c"], bins=np.arange(-32, 40, 2)), row=1, col=1)
fig.add_trace(go.Bar(x=rain_labels, y=rain_share.values, marker_color=[MUTED] + [BLUE] * 6,
                     text=[f"{s:.1f}%" for s in rain_share.values], textposition="outside"), row=1, col=2)
fig.add_trace(go.Bar(x=snow_counts.index, y=snow_counts.values, marker_color=[BLUE, RED],
                     text=[f"{n:,} hours" for n in snow_counts.values], textposition="outside"), row=1, col=3)
fig.add_trace(go.Bar(x=cloud_share.index, y=cloud_share.values, marker_color=BLUE, width=1.6), row=2, col=1)
fig.add_trace(go.Bar(x=wm_share.values, y=wm_share.index, orientation="h", marker_color=BLUE,
                     text=[f"{s:.1f}%" if s >= 0.1 else "< 0.1%" for s in wm_share.values], textposition="outside"), row=2, col=2)
fig.add_trace(go.Bar(x=np.arange(1, len(wd_counts) + 1), y=wd_counts.values,
                     marker_color=[MUTED if n < 50 else BLUE for n in wd_counts.values],
                     customdata=wd_counts.index, hovertemplate="%{customdata}: %{y:,} hours<extra></extra>"), row=2, col=3)
fig.update_xaxes(title_text="°C", row=1, col=1)
fig.update_yaxes(title_text="Hours", row=1, col=1)
fig.update_xaxes(title_text="mm in the hour", row=1, col=2)
fig.update_yaxes(title_text="% of hours (log)", type="log", dtick=1, range=[-1.2, 2.5], row=1, col=2)
fig.update_yaxes(title_text="Hours", range=[0, snow_counts.max() * 1.25], row=1, col=3)
fig.update_xaxes(title_text="% cloud cover", row=2, col=1)
fig.update_yaxes(title_text="% of hours", row=2, col=1)
fig.update_xaxes(title_text="% of hours", range=[0, wm_share.max() * 1.3], row=2, col=2)
fig.update_xaxes(title_text=f"{len(wd_counts)} descriptions, most to least frequent", row=2, col=3)
fig.update_yaxes(title_text="Hours (log)", type="log", dtick=1, row=2, col=3)
show_fig(fig, "Weather inputs are zero-inflated, discrete or imbalanced", "ts_inputs_dist", height=760, showlegend=False)

In [33]:
hol_days = hol_name.resample("D").first()
print(f"temp_c: 5–95% range {w['temp_c'].quantile(0.05):.0f} to {w['temp_c'].quantile(0.95):.0f} °C")
print(f"rain_1h: {(w['rain_1h'] == 0).mean():.1%} zeros; rainy hours: median {w.loc[w['rain_1h'] > 0, 'rain_1h'].median():.2f} mm")
print(f"snow: {snow_counts.iloc[0]:,} hours labelled Snow, only {snow_counts.iloc[1]} hours with snow_1h > 0")
print(f"clouds_all: {w['clouds_all'].nunique()} values, the 4 most frequent cover {w['clouds_all'].value_counts(normalize=True).head(4).sum():.0%}")
print(f"weather_main: {len(wm_share)} groups, two largest {', '.join(wm_share.index[-2:])} cover {wm_share.iloc[-2:].sum():.0f}%, "
      f"groups under 1%: {', '.join(wm_share[wm_share < 1].index)}")
print(f"weather_description: {len(wd_counts)} values, {(wd_counts < 50).sum()} seen in fewer than 50 hours")
print(f"holiday: {int(hol_days.ne('None').sum())} holiday days ({hol_days[hol_days.ne('None')].nunique()} names) "
      f"out of {len(hol_days):,} days = {hol_days.ne('None').mean():.1%}")
w[["temp_c", "rain_1h", "snow_1h", "clouds_all"]].corrwith(w[TARGET]).rename(f"Pearson r with {TARGET}").to_frame()

temp_c: 5–95% range -11 to 26 °C
rain_1h: 97.8% zeros; rainy hours: median 0.90 mm
snow: 844 hours labelled Snow, only 10 hours with snow_1h > 0
clouds_all: 30 values, the 4 most frequent cover 79%
weather_main: 10 groups, two largest Clear, Clouds cover 69%, groups under 1%: Smoke, Fog
weather_description: 34 values, 16 seen in fewer than 50 hours
holiday: 20 holiday days (11 names) out of 700 days = 2.9%


,Pearson r with traffic_volume
temp_c,0.127
rain_1h,-0.020
snow_1h,-0.000
clouds_all,0.106


Từ Hình 12 và các thống kê ở trên, các biến thời tiết trên train có hình dạng rất khác nhau:

| Biến | Hình dạng |
|---|---|
| `temp_c` | liên tục, **hai đỉnh theo mùa**, 5–95%: −11 đến 26 °C |
| `rain_1h` | **97.8% bằng 0**; khi có mưa thì lệch phải (trung vị 0.90 mm) |
| `snow_1h` | chỉ **10 giờ khác 0**, trong khi **844 giờ** có `weather_main` là `Snow`; cột này gần như không đo được tuyết |
| `clouds_all` | 30 giá trị nhưng 4 giá trị chiếm 79%; gần như rời rạc |
| `weather_main` | 10 nhóm; Clear và Clouds chiếm 69%, Smoke và Fog dưới 1% |
| `weather_description` | 34 giá trị, 16 giá trị xuất hiện dưới 50 giờ; đuôi dài, về sau có thể gặp giá trị mới |
| `holiday` | 20 ngày lễ trên 700 ngày (2.9%), 11 tên; sự kiện hiếm |

Tương quan tuyến tính giữa các biến thời tiết dạng số và nhãn đều nhỏ (|r| ≤ 0.13). Con số này **chưa đủ để nói thời tiết không có tác dụng**: giờ trong ngày chi phối nhãn rất mạnh, nên một tác động có thật nhưng ít khi xảy ra (như tuyết) vẫn có thể bị lấp đi.

Tương quan thô ở trên bị lịch làm nhiễu. Ví dụ sương mù hay xuất hiện lúc rạng sáng, mà giờ đó đường vốn đã vắng. Vì thế chúng tôi đo tác động bằng độ lệch so với mức thường lệ của cùng thứ, cùng giờ, chỉ trên các giờ ban ngày (6h–20h), không trùng ngày lễ và có dữ liệu thật.

In [34]:
base = eda[hol_name.eq("None").to_numpy() & (eda["is_filled"] == 0).to_numpy()].copy()
base["expected"] = base.groupby(base.index.dayofweek * 24 + base.index.hour)[TARGET].transform("median")
day = base[(base.index.hour >= 6) & (base.index.hour <= 20)]
day = day.assign(rel=(day[TARGET] / day["expected"] - 1) * 100)


def effect_table(groups, min_n=30):
    """Mean % deviation from the same weekday-hour, its 95% confidence half-width and the count, per group."""
    t = day.groupby(groups)["rel"].agg(["mean", "std", "count"]).rename(columns={"count": "n"})
    t = t[t["n"] >= min_n]
    return t.assign(ci95=1.96 * t["std"] / np.sqrt(t["n"])).sort_values("mean")


print(f"{len(day):,} daytime hours compared")
effect_table(day["weather_main"])[["mean", "ci95", "n"]].T

9,516 daytime hours compared


weather_main,Snow,Thunderstorm,Mist,Rain,Haze,Fog,Drizzle,Clear,Clouds
mean,-6.929,-3.610,-2.733,-2.453,-1.182,-0.946,-0.878,-0.376,-0.351
ci95,1.223,2.124,0.981,0.726,1.380,2.056,1.277,0.357,0.335
n,496.000,123.000,613.000,"1,104.000",197.000,52.000,156.000,"2,869.000","3,902.000"


In [35]:
rs = day[day["weather_main"].isin(["Rain", "Snow", "Drizzle", "Thunderstorm"])]
eff_desc = effect_table(rs["weather_description"], min_n=20)
group_of = rs.groupby("weather_description")["weather_main"].first()
fig = go.Figure()
for g_name, color in {"Rain": BLUE, "Snow": RED, "Drizzle": AQUA, "Thunderstorm": YELLOW}.items():
    part = eff_desc[group_of.reindex(eff_desc.index) == g_name]
    fig.add_trace(go.Scatter(x=part["mean"], y=part.index, mode="markers", name=g_name,
                             marker=dict(color=color, size=np.clip(np.sqrt(part["n"]) * 1.4, 7, 30), line=dict(color="#ffffff", width=1.5)),
                             error_x=dict(type="data", array=part["ci95"], color=color, thickness=1.2, width=0),
                             customdata=part["n"], hovertemplate="%{y}: %{x:+.1f}% (n=%{customdata:,})<extra></extra>"))
fig.add_trace(go.Scatter(x=eff_desc["mean"] + eff_desc["ci95"] + 0.6, y=eff_desc.index, mode="text", showlegend=False,
                         text=[f"{m:+.0f}% ({n:,} h)" for m, n in zip(eff_desc["mean"], eff_desc["n"])],
                         textposition="middle right", textfont=dict(color=INK_2, size=11), hoverinfo="skip"))
fig.add_vline(x=0, line_color=INK_2, line_width=1)
fig.add_annotation(x=1, y=-0.16, xref="paper", yref="paper", showarrow=False, xanchor="right", font=dict(color=INK_2, size=11),
                   text="dot = mean change, bar = 95% CI, dot size and (… h) = number of daytime hours")
snow_eff = eff_desc.loc[group_of.reindex(eff_desc.index) == "Snow", "mean"]
show_fig(fig, f"Intensity matters: '{snow_eff.idxmin()}' cuts daytime traffic by {-snow_eff.min():.0f}%, "
              f"'{snow_eff.idxmax()}' by {-snow_eff.max():.0f}% (dot size ∝ √n, bars = 95% CI)", "ts_weather_intensity",
         height=520, margin=dict(b=90),
         yaxis=dict(categoryorder="array", categoryarray=list(eff_desc.index)),
         xaxis=dict(title="Mean deviation from the same weekday-hour (%)",
                    range=[(eff_desc['mean'] - eff_desc['ci95']).min() - 1, (eff_desc['mean'] + eff_desc['ci95']).max() + 9]))
heavy = day["weather_description"].str.contains("heavy", case=False)
print(f"all 'heavy' rain/snow hours: {day.loc[heavy, 'rel'].mean():+.1f}% (n={heavy.sum()})")

all 'heavy' rain/snow hours: -10.1% (n=126)


Trên 9 516 giờ ban ngày, phần lớn các loại thời tiết chỉ làm lưu lượng lệch dưới 4%. Ngoại lệ là **tuyết, làm giảm trung bình 6.9%** (496 giờ, khoảng tin cậy 95% ±1.2 điểm phần trăm). Hình 13 cho thấy mức giảm **phụ thuộc vào cường độ**: tuyết nhẹ giảm 5%, **tuyết vừa (`snow`) giảm 21%**, mưa lớn giảm 9%, còn mưa nhẹ và mưa vừa hầu như không ảnh hưởng. Tuyết dày có quá ít giờ ban ngày trong train nên không đo được.

*Gợi ý:* thời tiết ít khi quan trọng, nhưng khi vượt một ngưỡng cường độ nào đó thì tác động lớn. Thông tin về cường độ chỉ có trong `weather_description`, vì `snow_1h` gần như không đo được tuyết (5.6).

### 5.7 Tác động của ngày lễ

Chúng tôi so tổng lưu lượng của mỗi ngày lễ với một ngày bình thường, lấy bằng trung vị của cùng thứ trong bốn tuần trước và bốn tuần sau (bỏ qua các ngày lễ).

In [36]:
rows = []
for d in day_total.index[is_hol_day]:
    same_dow = [d + pd.Timedelta(weeks=k) for k in range(-4, 5) if k != 0]
    same_dow = [x for x in same_dow if x in day_total.index and not is_hol_day[x]]
    rows.append({"date": d, "holiday": holiday_by_day[d], "actual": day_total[d], "expected": day_total[same_dow].median()})
hol = pd.DataFrame(rows).assign(change=lambda t: t["actual"] / t["expected"] - 1).sort_values("change")
by_name = hol.groupby("holiday").agg(mean_change=("change", "mean"), days=("change", "size")).sort_values("mean_change")
MAJOR_CUT = -0.30                                                      # [derived] wide gap between -33% and -14% in the plot below
is_major = by_name["mean_change"] <= MAJOR_CUT

# one bar per holiday name = average change over its occurrences; one dot per occurrence (2016, 2017)
fig = go.Figure()
fig.add_trace(go.Bar(y=by_name.index, x=by_name["mean_change"] * 100, orientation="h",
                     marker_color=np.where(is_major, RED, AQUA), opacity=0.85, showlegend=False,
                     hovertemplate="%{y}: average %{x:+.1f}%<extra></extra>"))
fig.add_trace(go.Scatter(y=hol["holiday"], x=hol["change"] * 100, mode="markers", name="each occurrence (2016, 2017)",
                         marker=dict(color="#ffffff", size=8, line=dict(color=INK_2, width=1.5)),
                         customdata=hol["date"].dt.strftime("%a %d/%m/%Y"),
                         hovertemplate="%{customdata}: %{x:+.1f}%<extra></extra>"))
fig.add_trace(go.Scatter(y=by_name.index, x=[9] * len(by_name), mode="text", showlegend=False, hoverinfo="skip",
                         text=[f"{v:+.0%}" for v in by_name["mean_change"]], textfont=dict(color=INK_2, size=12)))
fig.add_annotation(x=9, y=-0.9, text="average", showarrow=False, font=dict(color=INK_2, size=11))
for name, color in [("major holiday (average drop of 30% or more)", RED), ("other holiday", AQUA)]:
    fig.add_trace(go.Bar(y=[None], x=[None], name=name, marker_color=color, orientation="h"))
fig.add_vline(x=MAJOR_CUT * 100, line_dash="dash", line_color=INK_2,
              annotation_text="threshold −30%", annotation_position="top", annotation_font_color=INK_2)
fig.add_vline(x=0, line_color=INK_2, line_width=1)
show_fig(fig, f"Holidays fall into two groups: {int(is_major.sum())} major holidays cut daily traffic by "
              f"{-by_name.loc[is_major, 'mean_change'].max():.0%}–{-by_name['mean_change'].min():.0%}, "
              f"the others by at most {-by_name.loc[~is_major, 'mean_change'].min():.0%}", "ts_holiday_each",
         height=520, margin=dict(l=210), barmode="overlay",
         xaxis=dict(title="Daily traffic vs an ordinary day of the same weekday (%)", range=[-58, 12], tickvals=[-50, -40, -30, -20, -10, 0], ticksuffix="%"),
         yaxis=dict(autorange="reversed"))
print(f"mean change on a holiday: {hol['change'].mean():+.0%}")
by_name.style.format({"mean_change": "{:+.1%}"})

mean change on a holiday: -23%


,mean_change,days
holiday,,
Thanksgiving Day,-48.4%,2
Independence Day,-41.3%,2
New Years Day,-41.0%,2
Memorial Day,-38.0%,2
Labor Day,-33.8%,2
Christmas Day,-33.1%,1
Martin Luther King Jr Day,-13.8%,1
Washingtons Birthday,-8.1%,2
Veterans Day,-2.7%,2


Tính trung bình, ngày lễ thấp hơn ngày bình thường **23%**, nhưng con số này che đi khác biệt lớn giữa các ngày (Hình 14, bảng trên). Tính theo tên, sáu ngày lễ (Lễ Tạ ơn, Quốc khánh, Năm mới, Memorial Day, Labor Day, Giáng sinh) giảm **33–48%**. Martin Luther King Jr Day và Washington's Birthday giảm 8–14%, Veterans Day và Columbus Day gần như không giảm, còn mấy ngày State Fair lại tăng khoảng 3%. Giữa nhóm đầu (giảm ≥ 33%) và nhóm sau (giảm ≤ 14%) có một khoảng cách đủ rộng để tách làm hai.

Cần lưu ý cỡ mẫu: mỗi ngày lễ chỉ xuất hiện **1–2 lần** trong train, nên số liệu của từng ngày còn thô. Chúng tôi tách nhóm dựa vào khoảng cách lớn giữa hai nhóm, không dựa vào con số chính xác của từng ngày, và sẽ thử lại cách tách này trên một ngày lễ mô hình chưa thấy ở mục 8.

*Gợi ý:* nếu chỉ dùng một cờ ngày lễ thì các tác động rất khác nhau sẽ bị gộp làm một; cần tách riêng nhóm ngày lễ lớn. Ngày lễ chỉ chiếm khoảng 3% số ngày, quá hiếm để mô hình tự học được từ lag.

In [37]:
MAJOR_HOLIDAYS = by_name.index[by_name["mean_change"] <= -0.30].tolist()   # [derived] the six holidays that cut traffic by a third or more
MAJOR_HOLIDAYS

['Thanksgiving Day',
 'Independence Day',
 'New Years Day',
 'Memorial Day',
 'Labor Day',
 'Christmas Day']

### 5.8 Hình dạng cuối cùng của dữ liệu

Sau tiền xử lý và phân tích khám phá, dữ liệu mà mô hình phải học có các đặc điểm sau:

| Khía cạnh | Đặc điểm | Bằng chứng |
|---|---|---|
| Xu hướng | không có | trung bình 28 ngày trong 2 850 – 3 520 xe/giờ; xu hướng 1% phương sai (5.1, 5.2) |
| Chu kỳ | hai chu kỳ lồng nhau, ngày và tuần; giờ × thứ không cộng tính | giờ trong tuần giải thích 94% phương sai; cuối tuần đổi hình dạng ngày (5.1, 5.2) |
| Phụ thuộc quá khứ | mạnh ở 1–2 giờ trước, cùng giờ hôm qua, tuần trước | ACF lag 168 = 0.93, lag 1 = 0.91; PACF lag 1–2 (5.5) |
| Phần còn lại | tập trung quanh mức thường lệ, phân tán tăng theo mức, đuôi trái dày | 65% trong ±10%; kurtosis 28; giờ vắng nhiễu hơn theo % (5.3) |
| Ngoại lai | sự kiện thật (ngày lễ, bão tuyết, đóng đường) | median/MAD bắt 14/14 giờ đóng đường (5.4) |
| Thời tiết | hiếm, tác động mạnh khi vượt ngưỡng cường độ | tuyết vừa −21%, các loại khác trong ±4% (5.6) |
| Ngày lễ | hiếm, hai nhóm tác động khác hẳn nhau | −33% đến −48% so với ≤ −14% (5.7) |

Mục 6 và 7 dựa vào bảng này để chọn đặc trưng, metric và mô hình.

<a id="s6"></a>
## 6. Đặc trưng, mã hóa và chuẩn hóa

### 6.1 Tạo đặc trưng

Mỗi dòng tại thời điểm $t$ chỉ dùng thông tin có tại $t-1$ (ràng buộc ở mục 2). Mỗi nhóm đặc trưng ứng với một dòng của bảng 5.8:

| Nhóm | Đặc trưng | Căn cứ |
|---|---|---|
| Lag | `lag_1`, `lag_2`, `lag_24`, `lag_168` | phụ thuộc quá khứ (5.5) |
| Rolling | trung bình, độ lệch chuẩn của 24 giờ qua | mức và độ dao động của ngày gần nhất, giúp nhận ra ngày bất thường (5.3) |
| Lịch | giờ, thứ, tháng dạng sin/cos; `is_weekend` | chu kỳ và tương tác giờ × thứ (5.1, 5.2) |
| Ngày lễ | `is_holiday`, `is_major_holiday` | hai nhóm ngày lễ (5.7) |
| Thời tiết | 5 biến số và 2 biến phân loại **của giờ trước** | tác động theo cường độ (5.6); thời tiết giờ trước là thứ đã đo được |

Chuỗi không có xu hướng (5.1) nên không cần đặc trưng xu hướng; nhãn giữ ở thang gốc (5.1). Giờ và tháng được mã hóa bằng sin/cos để 23h nằm cạnh 0h và tháng 12 nằm cạnh tháng 1, điều mà một số nguyên không làm được. Các dòng chưa đủ 168 giờ lịch sử và các dòng có nhãn là giá trị điền bị loại khỏi bảng mô hình.

In [38]:
ROLL_WINDOW = 24


def cyclical(values, period):
    angle = 2 * np.pi * np.asarray(values) / period
    return np.sin(angle), np.cos(angle)


def make_features(ts, horizon=HORIZON, lags=None):
    """Feature table for every hour of `ts`, using only what is known `horizon` hours earlier.
    The target column holds observed values only; `hour_of_week` is kept aside for the linear model of 7.3."""
    lags = LAGS if lags is None else lags
    assert min(lags) >= horizon, "a lag shorter than the horizon is not known at forecast time"
    f = pd.DataFrame(index=ts.index)
    y = ts[TARGET]
    for k in lags:
        f[f"lag_{k}"] = y.shift(k)
    past = y.shift(horizon)
    f[f"roll_mean_{ROLL_WINDOW}"], f[f"roll_std_{ROLL_WINDOW}"] = past.rolling(ROLL_WINDOW).mean(), past.rolling(ROLL_WINDOW).std()
    idx = ts.index
    f["hour_sin"], f["hour_cos"] = cyclical(idx.hour, 24)
    f["dow_sin"], f["dow_cos"] = cyclical(idx.dayofweek, 7)
    f["month_sin"], f["month_cos"] = cyclical(idx.month - 1, 12)
    f["is_weekend"] = (idx.dayofweek >= 5).astype(int)
    holiday = holiday_of(idx)
    f["is_holiday"] = holiday.ne("None").astype(int)
    f["is_major_holiday"] = holiday.isin(MAJOR_HOLIDAYS).astype(int)
    for c in [*NUM_EXOG, *CAT_EXOG]:
        f[f"{c}_lag{horizon}"] = ts[c].shift(horizon)
    f["hour_of_week"] = (idx.dayofweek * 24 + idx.hour).astype(str)
    f[TARGET] = y.where(ts["is_filled"].eq(0))       # filled hours are inputs, never targets
    return f


features = make_features(ts)
CAT_FEATURES = [f"{c}_lag{HORIZON}" for c in CAT_EXOG]
NUM_FEATURES = [c for c in features.columns if c not in [TARGET, "hour_of_week", *CAT_FEATURES]]
FEATURE_COLS = NUM_FEATURES + CAT_FEATURES
model_df = features.dropna(subset=[*NUM_FEATURES, TARGET])

t = model_df.index
train, val, test = (model_df[t <= TRAIN_END], model_df[(t > TRAIN_END) & (t <= VAL_END)], model_df[t > VAL_END])
assert train.index.max() < val.index.min() and val.index.max() < test.index.min()
print(f"{len(features):,} hours -> {len(model_df):,} model rows | {len(NUM_FEATURES)} numeric + {len(CAT_FEATURES)} categorical features")
pd.DataFrame({"rows": [len(p) for p in (train, val, test)], "share": [len(p) / len(model_df) for p in (train, val, test)]},
             index=["train", "validation", "test"])

24,096 hours -> 22,969 model rows | 20 numeric + 2 categorical features


,rows,share
train,15696,0.683
validation,3607,0.157
test,3666,0.160


### 6.2 Mã hóa và chuẩn hóa

Ở mục 7 sẽ có cả mô hình tuyến tính lẫn mô hình cây, nên bước biến đổi phải dùng được cho cả hai:

- **Biến phân loại** (`weather_main`, `weather_description`) có nhiều giá trị hiếm và về sau có thể gặp giá trị mới (5.6). Vì vậy dùng `OneHotEncoder(handle_unknown="ignore")`: gặp giá trị lạ thì ra vector 0 thay vì báo lỗi.
- **Biến số** có thang đo chênh nhau nhiều (xe, °C, mm, %), nên dùng `StandardScaler`. Mô hình tuyến tính cần bước này, còn mô hình cây thì không bị ảnh hưởng.

Cả hai bước nằm trong `Pipeline` của từng mô hình, nên chỉ được fit trên dữ liệu huấn luyện, kể cả bên trong từng fold cross-validation.

In [39]:
preprocessor = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), NUM_FEATURES),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                      ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), CAT_FEATURES)])
X_train, y_train = train[FEATURE_COLS], train[TARGET]
X_val, y_val = val[FEATURE_COLS], val[TARGET]
X_test, y_test = test[FEATURE_COLS], test[TARGET]
print(f"{len(FEATURE_COLS)} features -> {len(clone(preprocessor).fit(X_train).get_feature_names_out())} model columns after One-Hot")
corr = train[[TARGET, *NUM_FEATURES]].corr()
display(corr[TARGET].drop(TARGET).sort_values(key=np.abs, ascending=False).head(6).rename(f"r with {TARGET} (train)").to_frame().T)
print(f"corr(lag_1, lag_2) = {corr.loc['lag_1', 'lag_2']:.2f}; corr(hour_sin, hour_cos) is 0 by construction, but they form one pair")

22 features -> 64 model columns after One-Hot


,lag_168,lag_1,lag_24,hour_cos,lag_2,hour_sin
r with traffic_volume (train),0.949,0.899,0.865,-0.767,0.709,-0.245


corr(lag_1, lag_2) = 0.91; corr(hour_sin, hour_cos) is 0 by construction, but they form one pair


Trên train, `lag_168`, `lag_1` và `lag_24` có tương quan tuyến tính với nhãn cao nhất, sau đó là `hour_cos`; thứ tự này giống với ACF ở 5.5. `lag_1` và `lag_2` tương quan rất cao với nhau, nên mô hình tuyến tính cần phạt L2 để hệ số không dao động, và khi đo tầm quan trọng thì phải đo theo nhóm (8.4).

<a id="s7"></a>
## 7. Metric và mô hình

### 7.1 Chọn metric

Nhãn là số đếm nên chất lượng dự báo được đo bằng sai số giữa giá trị dự báo và giá trị thật; các metric phân loại như accuracy hay confusion matrix không dùng được ở đây. Bảng dưới xem xét bốn metric hồi quy thường gặp dựa trên hình dạng dữ liệu ở mục 5:

| Metric | Công thức | Nhận xét theo dữ liệu |
|---|---|---|
| MAE | $\frac{1}{n}\sum \lvert y - \hat{y} \rvert$ | cùng đơn vị với nhãn (xe/giờ); mỗi giờ đóng góp đúng bằng số xe bị lệch |
| RMSE | $\sqrt{\frac{1}{n}\sum (y - \hat{y})^2}$ | vì đuôi dày (5.3), vài giờ khó nhất có thể chi phối |
| MAPE | $\frac{100\%}{n}\sum \left\lvert \frac{y - \hat{y}}{y} \right\rvert$ | giờ vắng nhiễu hơn theo tỷ lệ (5.3) nên dễ bị đẩy lên; không xác định khi $y = 0$ |
| $R^2$ | $1 - \frac{\sum (y - \hat{y})^2}{\sum (y - \bar{y})^2}$ | lịch đã giải thích 94% phương sai (5.2), nên mô hình nào tạm ổn cũng có $R^2$ cao |

Hai nhận xét về RMSE và MAPE được kiểm tra trực tiếp: Hình 15 cho thấy mỗi metric lấy trọng số từ những giờ nào trong ngày. Lúc này chưa có mô hình nào, nên chúng tôi dùng sai số của dự báo đơn giản nhất ("giờ tới bằng giờ vừa rồi") trên train, rồi kiểm tra lại với mô hình cuối ở 8.3.

In [40]:
def metric_weights_by_hour(err, y_obs):
    """Share of MAE, of the squared error and of MAPE that comes from each hour of the day."""
    by_hour = pd.DataFrame({"abs": err.abs(), "pct": err.abs() / y_obs * 100, "sq": err ** 2}).groupby(err.index.hour).mean()
    return pd.DataFrame({"MAE": by_hour["abs"] / by_hour["abs"].sum(), "RMSE (squared error)": by_hour["sq"] / by_hour["sq"].sum(),
                         "MAPE": by_hour["pct"] / by_hour["pct"].sum()}) * 100


NIGHT = [0, 1, 2, 3, 4, 22, 23]
tr = ts.loc[:TRAIN_END]
naive_err = (tr[TARGET] - ts[TARGET].shift(HORIZON).reindex(tr.index))[tr["is_filled"].eq(0) & tr[TARGET].gt(0)].dropna()
share = metric_weights_by_hour(naive_err, tr[TARGET].reindex(naive_err.index))
night_share = pd.Series({"share of hours": len(NIGHT) / 24 * 100, **{m: share.loc[NIGHT, m].sum() for m in share.columns}})

fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.1, column_widths=[0.62, 0.38],
                    subplot_titles=("Share of each metric coming from each hour (naive forecast, train)", "Weight of the night hours (22h–4h)"))
for m, color in [("MAE", BLUE), ("RMSE (squared error)", MUTED), ("MAPE", ORANGE)]:
    fig.add_trace(go.Scatter(x=share.index, y=share[m], mode="lines+markers", name=m, line=dict(color=color, width=2.5)), row=1, col=1)
fig.add_vrect(x0=-0.5, x1=4.5, fillcolor=MUTED, opacity=0.18, line_width=0, row=1, col=1, layer="below",
              annotation_text="night", annotation_position="top left", annotation_font_color=INK_2)
fig.add_vrect(x0=21.5, x1=23.5, fillcolor=MUTED, opacity=0.18, line_width=0, row=1, col=1, layer="below",
              annotation_text="night", annotation_position="top right", annotation_font_color=INK_2)
fig.add_vrect(x0=4.5, x1=7.5, fillcolor=YELLOW, opacity=0.15, line_width=0, row=1, col=1, layer="below",
              annotation_text="morning ramp-up", annotation_position="top left", annotation_font_color=INK_2)
fig.add_trace(go.Bar(x=night_share.index, y=night_share.values, showlegend=False, marker_color=[YELLOW, BLUE, MUTED, ORANGE],
                     text=[f"{v:.0f}%" for v in night_share.values], textposition="outside"), row=1, col=2)
fig.update_xaxes(title_text="Hour of day", dtick=3, row=1, col=1)
fig.update_yaxes(title_text="% of the metric", row=1, col=1)
fig.update_yaxes(title_text="% of the total", range=[0, night_share.max() * 1.2], row=1, col=2)
show_fig(fig, f"Night hours are {night_share['share of hours']:.0f}% of the day but {night_share['MAPE']:.0f}% of MAPE; "
              "squared error is dominated by the morning ramp-up", "ts_metric_by_hour", height=460,
         legend=dict(orientation="v", x=0.36, y=0.9, yanchor="top"))
print(f"share of the squared error from 5h–7h: {share.loc[[5, 6, 7], 'RMSE (squared error)'].sum():.0f}%")
night_share.round(1).rename("% (night hours 22h–4h)").to_frame().T

share of the squared error from 5h–7h: 50%


,share of hours,MAE,RMSE (squared error),MAPE
% (night hours 22h–4h),29.200,20.500,12.900,57.800


Giờ đêm chỉ chiếm 29% số giờ nhưng góp tới **58% MAPE**. Sai 300 xe lúc chỉ có 750 xe đã là 40%, trong khi sai 300 xe lúc 16h (khoảng 6 000 xe) chỉ là 5%. Sai số bình phương dồn vào đợt tăng vọt buổi sáng 5h–7h, nên RMSE phần lớn do vài giờ khó nhất quyết định. MAE chia trọng số đều nhất (21% từ giờ đêm), theo đúng số xe bị lệch (Hình 15).

Từ đó chúng tôi quyết định:
- **MAE là metric chính**, dùng để chọn mô hình trên validation.
- RMSE được báo cáo kèm; tỷ số RMSE/MAE cho biết mô hình có hay sai lớn không. MAPE chỉ để tham khảo.
- Không dùng $R^2$ để chọn mô hình. Thay vào đó, mỗi mô hình được so với các mốc đơn giản ở 7.2 qua **skill** $= 1 - \text{MAE}_\text{model} / \text{MAE}_\text{mốc}$ (dương nghĩa là tốt hơn mốc). Mốc khó vượt nhất là "cùng giờ tuần trước" (5.2), nên skill so với mốc này là con số chúng tôi quan tâm nhất.
- Mọi metric chỉ tính trên giờ có dữ liệu thật.

### 7.2 Mốc so sánh và mô hình ứng viên

Hai mốc so sánh lấy thẳng từ mục 5:
- **Naive** $\hat y_t = y_{t-1}$: giờ tới bằng giờ vừa rồi, dựa trên ACF lag 1 = 0.91.
- **Seasonal naive** $\hat y_t = y_{t-24}$ hoặc $y_{t-168}$: lặp lại cùng giờ hôm qua hoặc tuần trước, dựa trên hai chu kỳ ở 5.2.

Mỗi mô hình ứng viên được chọn để kiểm tra một giả thuyết về hình dạng dữ liệu (bảng 5.8). Mô hình tuyến tính được thử với **hai cách biểu diễn lịch**, để tách hai câu hỏi: quan hệ có tuyến tính không, và lịch đã được biểu diễn đủ chưa.

| Mô hình | Giả thuyết được kiểm tra | Phù hợp với dữ liệu vì | Điểm yếu dự kiến |
|---|---|---|---|
| Ridge (lịch sin/cos) | tuyến tính, với cùng đặc trưng như mô hình cây, là đủ | phạt L2 ổn định khi các lag tương quan mạnh | một hàm tuyến tính của sin/cos không vẽ được ngày hai đỉnh; không có tương tác giờ × thứ |
| Ridge + one-hot giờ trong tuần | tuyến tính là đủ nếu biết mức thường lệ của từng giờ trong tuần | 168 biến giả mô tả đúng hình dạng tuần (5.2), kể cả tương tác giờ × thứ | tác động của lag, thời tiết, ngày lễ vẫn cộng tính và như nhau ở mọi giờ |
| Random Forest, HistGradientBoosting | cần quan hệ phi tuyến và tương tác | cây tự chia theo giờ, thứ và ngưỡng thời tiết | ngoại suy kém ngoài vùng đã thấy |
| SARIMAX + Fourier | mô hình thống kê cổ điển cho chuỗi có chu kỳ là đủ | không cần sai phân (5.5); Fourier mô tả hai chu kỳ bằng ít tham số; phần AR mô tả trí nhớ ngắn hạn | khó đưa thời tiết phân loại vào; fit chậm |

Để so sánh cho công bằng, cả hai mô hình cây đều được tinh chỉnh theo cùng một cách (7.4).

### 7.3 So sánh trên tập validation

In [41]:
def regression_scores(y_true, y_pred):
    y_pred = np.asarray(y_pred, dtype=float)
    keep = ~np.isnan(y_pred)                  # a baseline lag can fall into a long gap -> no forecast
    y_true, y_pred = y_true.to_numpy()[keep], y_pred[keep]
    err, nonzero = y_true - y_pred, y_true != 0
    return {"MAE": np.mean(np.abs(err)), "RMSE": np.sqrt(np.mean(err ** 2)),
            "MAPE %": np.mean(np.abs(err[nonzero] / y_true[nonzero])) * 100, "R2": r2_score(y_true, y_pred)}


BASELINES = {f"Naive (t-{HORIZON})": HORIZON, **{f"Seasonal naive (t-{p})": p for p in SEASONAL_PERIODS}}
NAIVE, WEEKLY = f"Naive (t-{HORIZON})", "Seasonal naive (t-168)"


def baseline_forecast(index, lag):
    return ts[TARGET].shift(lag).reindex(index).to_numpy()


def make_model(estimator):
    return Pipeline([("prep", clone(preprocessor)), ("model", estimator)])


MODELS = {"Ridge": Ridge(alpha=1.0),
          "Random Forest": RandomForestRegressor(n_estimators=200, min_samples_leaf=3, n_jobs=-1, random_state=SEED),
          "HistGradientBoosting": HistGradientBoostingRegressor(random_state=SEED)}

fitted, val_pred = {}, {}
for name, lag in BASELINES.items():
    val_pred[name] = baseline_forecast(y_val.index, lag)
for name, estimator in MODELS.items():
    fitted[name] = make_model(estimator).fit(X_train, y_train)
    val_pred[name] = fitted[name].predict(X_val)

# Linear model with the weekly shape given explicitly: one dummy per hour of the week (168 columns)
LIN_HOW = "Ridge + hour-of-week one-hot"
LIN_COLS = FEATURE_COLS + ["hour_of_week"]
lin_prep = ColumnTransformer([
    ("num", StandardScaler(), NUM_FEATURES),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), [*CAT_FEATURES, "hour_of_week"])])
lin_how = Pipeline([("prep", lin_prep), ("model", Ridge(alpha=1.0))]).fit(train[LIN_COLS], y_train)
val_pred[LIN_HOW] = lin_how.predict(val[LIN_COLS])


def results_table():
    out = pd.DataFrame({name: regression_scores(y_val, p) for name, p in val_pred.items()}).T
    out["skill vs naive"] = 1 - out["MAE"] / out.loc[NAIVE, "MAE"]
    out["skill vs t-168"] = 1 - out["MAE"] / out.loc[WEEKLY, "MAE"]
    return out.sort_values("MAE")


results_table()

,MAE,RMSE,MAPE %,R2,skill vs naive,skill vs t-168
HistGradientBoosting,188.716,303.065,8.448,0.976,0.676,0.578
Ridge + hour-of-week one-hot,188.949,308.744,9.373,0.976,0.676,0.578
Random Forest,200.174,341.328,8.421,0.970,0.656,0.553
Ridge,329.055,465.887,18.385,0.944,0.435,0.265
Seasonal naive (t-168),447.638,779.547,17.896,0.844,0.232,0.000
Naive (t-1),582.573,803.189,27.427,0.834,0.000,-0.301
Seasonal naive (t-24),627.288,"1,065.484",28.835,0.709,-0.077,-0.401


**SARIMAX với số hạng Fourier** (dynamic harmonic regression). Hình dạng tuần được mô tả bằng $K$ cặp sin/cos, còn trí nhớ ngắn hạn do phần sai số AR($p$) đảm nhận:

$$y_t = c + \sum_{k=1}^{K}\left[a_k \sin\frac{2\pi k t}{168} + b_k \cos\frac{2\pi k t}{168}\right] + \beta^\top x_t + \eta_t, \qquad \eta_t \sim \text{AR}(p).$$

Các họa âm $k = 7, 14, 21, \dots$ có chu kỳ 24 giờ, nên chu kỳ ngày cũng nằm sẵn trong đó. $x_t$ gồm các cờ ngày lễ và tích của chúng với ba họa âm ngày đầu tiên, vì ngày lễ làm đổi hình dạng của cả ngày. Không lấy sai phân ($d = 0$, theo 5.5). Bậc $p \in \{1, 2, 3\}$ (PACF gợi ý 2) và số họa âm $K \in \{21, 42, 63\}$ được thử trên lưới; bảng ghi cả AIC (tính trên train) lẫn MAE validation. Dự báo vẫn là một bước, như các mô hình khác.

In [42]:
def sarimax_exog(index, K):
    """Weekly Fourier terms + holiday flags (+ holiday x daily harmonics) for every timestamp of `index`."""
    hrs = ((index - pd.Timestamp("2016-01-04")) / pd.Timedelta(hours=1)).to_numpy()   # origin: a Monday 00:00
    X = {}
    for k in range(1, K + 1):
        X[f"sin_{k}"], X[f"cos_{k}"] = np.sin(2 * np.pi * k * hrs / 168), np.cos(2 * np.pi * k * hrs / 168)
    X = pd.DataFrame(X, index=index)
    holiday = holiday_of(index)
    X["is_holiday"] = holiday.ne("None").astype(float)
    X["is_major_holiday"] = holiday.isin(MAJOR_HOLIDAYS).astype(float)
    for k in (7, 14, 21):
        X[f"hol_sin_{k}"], X[f"hol_cos_{k}"] = X["is_holiday"] * X[f"sin_{k}"], X["is_holiday"] * X[f"cos_{k}"]
    return X


y_sx = ts[TARGET]                                  # continuous series (short gaps filled from the past)
in_train, up_to_val = y_sx.index <= TRAIN_END, y_sx.index <= VAL_END
sarimax_rows, sarimax_pred = [], {}
for K in [21, 42, 63]:
    X_sx = sarimax_exog(y_sx.index, K)
    for p in [1, 2, 3]:
        start = time.time()
        fit = SARIMAX(y_sx[in_train], exog=X_sx[in_train], order=(p, 0, 0), trend="c").fit(disp=False, maxiter=200)
        sarimax_pred[(p, K)] = fit.apply(y_sx[up_to_val], exog=X_sx[up_to_val], refit=False).fittedvalues.reindex(y_val.index)
        sarimax_rows.append({"p": p, "K": K, "AIC (train)": fit.aic, "AR(1)": fit.params["ar.L1"],
                             "validation MAE": regression_scores(y_val, sarimax_pred[(p, K)])["MAE"], "seconds": time.time() - start})
sarimax_table = pd.DataFrame(sarimax_rows).set_index(["K", "p"])
BEST_P, BEST_K = sarimax_table["validation MAE"].idxmin()[::-1]
SARIMAX_NAME = f"SARIMAX({BEST_P},0,0) + Fourier K={BEST_K}"
val_pred[SARIMAX_NAME] = sarimax_pred[(BEST_P, BEST_K)].to_numpy()
print(f"lowest AIC: p={sarimax_table['AIC (train)'].idxmin()[1]}, K={sarimax_table['AIC (train)'].idxmin()[0]} | "
      f"lowest validation MAE: p={BEST_P}, K={BEST_K}")
sarimax_table

lowest AIC: p=3, K=63 | lowest validation MAE: p=1, K=63


AIC (train)  AR(1)  validation MAE  seconds
K  p                                             
21 1  259,790.735  0.747         370.315   16.622
   2  259,454.625  0.853         356.783   18.713
   3  258,416.025  0.888         331.809   24.828
42 1  251,536.932  0.822         231.127   20.143
   2  251,082.197  0.688         230.975   29.704
   3  250,904.884  0.671         228.438   43.266
63 1  250,122.504  0.837         206.302   28.049
   2  249,742.255  0.711         208.078   40.764
   3  249,717.107  0.705         208.130   45.594

### 7.4 Tinh chỉnh siêu tham số với `TimeSeriesSplit`

Hai mô hình cây được tinh chỉnh bằng `GridSearchCV` theo cùng một cách: HistGradientBoosting theo learning rate và số lá, Random Forest theo tỷ lệ đặc trưng xét ở mỗi lần chia và số mẫu tối thiểu ở lá. Vì các quan sát phụ thuộc thời gian, cross-validation dùng `TimeSeriesSplit` thay cho $k$-fold xáo trộn: fold nào cũng học trên quá khứ và đánh giá trên giai đoạn liền sau.

In [43]:
GRIDS = {"HistGradientBoosting": {"model__learning_rate": [0.05, 0.1], "model__max_leaf_nodes": [15, 31, 63], "model__max_iter": [300]},
         "Random Forest": {"model__max_features": [0.33, 1.0], "model__min_samples_leaf": [1, 5]}}
searches = {}
for name, grid in GRIDS.items():
    start = time.time()
    searches[name] = GridSearchCV(make_model(MODELS[name]), grid, scoring="neg_mean_absolute_error",
                                  cv=TimeSeriesSplit(n_splits=3), n_jobs=-1).fit(X_train, y_train)
    fitted[f"{name} (tuned)"] = searches[name].best_estimator_
    val_pred[f"{name} (tuned)"] = searches[name].best_estimator_.predict(X_val)
    print(f"{name}: best {searches[name].best_params_} | CV MAE on train {-searches[name].best_score_:,.1f} | {time.time() - start:.0f} s")
results = results_table()
results

HistGradientBoosting: best {'model__learning_rate': 0.05, 'model__max_iter': 300, 'model__max_leaf_nodes': 31} | CV MAE on train 185.9 | 28 s
Random Forest: best {'model__max_features': 0.33, 'model__min_samples_leaf': 1} | CV MAE on train 180.0 | 109 s


,MAE,RMSE,MAPE %,R2,skill vs naive,skill vs t-168
HistGradientBoosting (tuned),184.650,299.218,8.254,0.977,0.683,0.588
Random Forest (tuned),185.721,304.807,8.115,0.976,0.681,0.585
HistGradientBoosting,188.716,303.065,8.448,0.976,0.676,0.578
Ridge + hour-of-week one-hot,188.949,308.744,9.373,0.976,0.676,0.578
Random Forest,200.174,341.328,8.421,0.970,0.656,0.553
"SARIMAX(1,0,0) + Fourier K=63",206.302,340.050,11.851,0.970,0.646,0.539
Ridge,329.055,465.887,18.385,0.944,0.435,0.265
Seasonal naive (t-168),447.638,779.547,17.896,0.844,0.232,0.000
Naive (t-1),582.573,803.189,27.427,0.834,0.000,-0.301
Seasonal naive (t-24),627.288,"1,065.484",28.835,0.709,-0.077,-0.401


In [44]:
# Every grid point of the two searches: mean CV MAE over the 3 time-ordered folds
tuning = pd.concat({name: pd.DataFrame({"parameters": [", ".join(f"{k.replace('model__', '')}={v}" for k, v in p.items())
                                                       for p in s.cv_results_["params"]],
                                        "CV MAE": -s.cv_results_["mean_test_score"],
                                        "std over folds": s.cv_results_["std_test_score"]}).sort_values("CV MAE")
                    for name, s in searches.items()})
tuning

parameters  \
HistGradientBoosting 1  learning_rate=0.05, max_iter=300, max_leaf_nod...   
                     2  learning_rate=0.05, max_iter=300, max_leaf_nod...   
                     0  learning_rate=0.05, max_iter=300, max_leaf_nod...   
                     3  learning_rate=0.1, max_iter=300, max_leaf_node...   
                     4  learning_rate=0.1, max_iter=300, max_leaf_node...   
                     5  learning_rate=0.1, max_iter=300, max_leaf_node...   
Random Forest        0              max_features=0.33, min_samples_leaf=1   
                     1              max_features=0.33, min_samples_leaf=5   
                     2               max_features=1.0, min_samples_leaf=1   
                     3               max_features=1.0, min_samples_leaf=5   

                        CV MAE  std over folds  
HistGradientBoosting 1 185.919          22.968  
                     2 189.067          24.481  
                     0 189.334          18.559  
                     3 189.711          23.328  
                     4 189.884          23.177  
                     5 191.605          25.195  
Random Forest        0 179.975          17.151  
                     1 185.679          18.416  
                     2 186.882          19.817  
                     3 190.895          22.206

Bảng trên liệt kê mọi điểm của hai lưới. Với HistGradientBoosting, cấu hình tốt nhất là learning rate 0.05 và 31 lá (CV MAE 185.4), nhưng sáu cấu hình chỉ chênh nhau 6 xe/giờ, nhỏ hơn nhiều so với độ lệch chuẩn giữa các fold (18–25 xe/giờ). Với Random Forest, xét 1/3 số đặc trưng ở mỗi lần chia giúp nhiều hơn (180.0 so với 186.9 khi xét tất cả), có thể vì các cây bớt giống nhau khi hai lag tương quan mạnh cùng có mặt.

In [45]:
ordered = results.sort_values("MAE", ascending=False)
fig = go.Figure(go.Bar(x=ordered["MAE"], y=ordered.index, orientation="h",
                       marker_color=[MUTED if n in BASELINES else (ORANGE if n == SARIMAX_NAME else BLUE) for n in ordered.index],
                       text=[f"{m:,.0f}  ({s:+.0%} vs t-168)" for m, s in zip(ordered["MAE"], ordered["skill vs t-168"])],
                       textposition="outside", textfont_color=INK_2))
show_fig(fig, f"Validation MAE (grey = baselines, orange = statistical model): best = {results['MAE'].idxmin()}",
         "ts_validation_mae", xaxis=dict(title="MAE (vehicles/hour)", range=[0, ordered["MAE"].max() * 1.35]),
         height=460, showlegend=False, margin=dict(l=230))

### 7.5 Kiểm định khác biệt giữa các mô hình

Bảng MAE chỉ cho một con số trên một giai đoạn validation. Để biết khoảng cách giữa hai mô hình có lớn hơn dao động ngẫu nhiên không, chúng tôi tính khoảng tin cậy 95% cho hiệu MAE bằng **block bootstrap**: lấy mẫu lại chuỗi hiệu sai số tuyệt đối theo từng khối giờ liên tiếp, 2 000 lần. Sai số tương quan cả trong ngày lẫn trong tuần (5.5), nên chúng tôi chạy với hai độ dài khối là 24 giờ và 168 giờ. Một khác biệt chỉ được tính là có ý nghĩa khi khoảng tin cậy không chứa 0 với **cả hai** độ dài khối.

In [46]:
def mae_diff_ci(y_true, pred_a, pred_b, block=24, n_boot=2000, seed=SEED):
    """MAE(a) − MAE(b) and its 95% moving-block bootstrap interval (negative = a is better)."""
    d = np.abs(np.asarray(y_true) - np.asarray(pred_a)) - np.abs(np.asarray(y_true) - np.asarray(pred_b))
    d = d[~np.isnan(d)]
    rng = np.random.default_rng(seed)
    starts = rng.integers(0, len(d) - block + 1, size=(n_boot, int(np.ceil(len(d) / block))))
    idx = (starts[:, :, None] + np.arange(block)).reshape(n_boot, -1)[:, :len(d)]
    lo, hi = np.percentile(d[idx].mean(axis=1), [2.5, 97.5])
    return d.mean(), lo, hi


def comparison_table(y_true, preds, pairs):
    rows = {}
    for a, b in pairs:
        diff, lo24, hi24 = mae_diff_ci(y_true, preds[a], preds[b], block=24)
        _, lo168, hi168 = mae_diff_ci(y_true, preds[a], preds[b], block=168)
        rows[f"{a}  vs  {b}"] = {"MAE difference": diff, "95% CI (24 h blocks)": f"[{lo24:.1f}; {hi24:.1f}]",
                                 "95% CI (168 h blocks)": f"[{lo168:.1f}; {hi168:.1f}]",
                                 "significant (both)": not (lo24 <= 0 <= hi24 or lo168 <= 0 <= hi168)}
    return pd.DataFrame(rows).T


BEST_NAME = results.loc[list(fitted)]["MAE"].idxmin()
comparison_table(y_val, val_pred, [(BEST_NAME, other) for other in results.index
                                   if other not in (BEST_NAME, NAIVE, "Seasonal naive (t-24)")])

,MAE difference,95% CI (24 h blocks),95% CI (168 h blocks),significant (both)
HistGradientBoosting (tuned) vs Random Forest (tuned),-1.071,[-6.6; 3.6],[-5.9; 3.0],False
HistGradientBoosting (tuned) vs HistGradientBoosting,-4.066,[-6.4; -1.8],[-6.5; -1.3],True
HistGradientBoosting (tuned) vs Ridge + hour-of-week one-hot,-4.299,[-14.4; 4.3],[-17.1; 5.9],False
HistGradientBoosting (tuned) vs Random Forest,-15.524,[-24.0; -7.7],[-24.8; -7.2],True
"HistGradientBoosting (tuned) vs SARIMAX(1,0,0) + Fourier K=63",-21.652,[-37.0; -8.0],[-39.9; -5.6],True
HistGradientBoosting (tuned) vs Ridge,-144.405,[-162.8; -127.0],[-163.8; -128.9],True
HistGradientBoosting (tuned) vs Seasonal naive (t-168),-262.988,[-317.3; -210.4],[-325.7; -208.2],True


### 7.6 Mô hình cuối

Mô hình học máy có MAE validation thấp nhất được fit lại trên train + validation (dữ liệu gần nhất là quan trọng nhất cho dự báo) và chấm đúng một lần trên test. SARIMAX chỉ được so trên validation, vì tập test chỉ dành cho mô hình được chọn.

In [47]:
final_model = clone(fitted[BEST_NAME]).fit(pd.concat([X_train, X_val]), pd.concat([y_train, y_val]))
print(f"final model: {BEST_NAME} | validation MAE {results.loc[BEST_NAME, 'MAE']:,.1f} | "
      f"skill vs naive {results.loc[BEST_NAME, 'skill vs naive']:+.1%}, vs t-168 {results.loc[BEST_NAME, 'skill vs t-168']:+.1%}")

final model: HistGradientBoosting (tuned) | validation MAE 184.7 | skill vs naive +68.3%, vs t-168 +58.8%


In [48]:
# Why is "same hour yesterday" worse than naive? MAE of each baseline by weekday on validation
wd_mae = pd.DataFrame({name: pd.Series(np.abs(y_val.to_numpy() - val_pred[name]), index=y_val.index)
                             .groupby(y_val.index.dayofweek).mean() for name in BASELINES})
wd_mae.index = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
wd_mae

,Naive (t-1),Seasonal naive (t-24),Seasonal naive (t-168)
Mon,601.045,"1,172.867",679.055
Tue,664.098,547.689,537.016
Wed,669.292,336.046,244.366
Thu,672.228,277.020,320.631
Fri,617.918,306.044,328.953
Sat,440.042,"1,088.108",594.283
Sun,423.856,633.860,419.556


Chúng tôi đọc kết quả validation (Hình 16 và các bảng trên) theo từng giả thuyết ở 7.2:

- **Mốc chu kỳ mạnh.** Trong các mốc, "cùng giờ tuần trước" tốt nhất (MAE 448). "Cùng giờ hôm qua" (627) còn thua naive (583). Bảng MAE theo thứ ở ô trên cho thấy lý do: t−24 sai 1 173 xe/giờ vào thứ Hai và 1 088 vào thứ Bảy, gấp đôi naive, vì nó lấy Chủ nhật để đoán thứ Hai và thứ Sáu để đoán thứ Bảy. Từ thứ Ba đến thứ Sáu, t−24 lại tốt hơn naive.
- **Biểu diễn lịch quan trọng hơn tính phi tuyến.** Ridge dùng sin/cos (MAE 329) kém xa các mô hình cây. Nhưng khi đổi sang one-hot giờ trong tuần, Ridge đạt **188.9**, khác HistGradientBoosting tinh chỉnh không có ý nghĩa thống kê (hiệu 3.3 xe/giờ, khoảng tin cậy chứa 0). Như vậy khoảng cách ban đầu là do sin/cos tuyến tính không vẽ lại được hình dạng tuần (5.1), chứ không phải do quan hệ phi tuyến. Lợi thế thật của mô hình cây là **tự học** được tương tác giờ × thứ mà không cần ai mô tả sẵn.
- **Hai mô hình cây ngang nhau khi được tinh chỉnh như nhau**: HistGradientBoosting 185.7, Random Forest 185.7 (hiệu 0.05 xe/giờ). Phần HistGradientBoosting hơn Random Forest mặc định (200.2) đều nhờ tinh chỉnh.
- **SARIMAX + Fourier là một mốc thống kê tốt** (206.3), thua mô hình tốt nhất 21 xe/giờ và khoảng tin cậy không chứa 0. Số họa âm $K$ ảnh hưởng lớn nhất (MAE 332–370 với $K = 21$, 206–208 với $K = 63$), vì các đỉnh cao điểm rất nhọn. Bậc AR ảnh hưởng rất ít: với $K = 63$, ba bậc chỉ chênh nhau dưới 2 xe/giờ. AIC chọn $p = 3$, validation chọn $p = 1$, còn PACF gợi ý 2.
- **$R^2$ không giúp phân biệt mô hình**: naive đã đạt 0.83, và mọi mô hình học máy đều trên 0.94.

Kết luận về ý nghĩa thống kê không đổi dù dùng khối bootstrap 24 giờ hay 168 giờ. Theo quy tắc đã đặt từ trước (MAE validation thấp nhất), **HistGradientBoosting tinh chỉnh** là mô hình cuối. Cần nói rõ rằng ba mô hình đứng đầu (HistGradientBoosting, Random Forest tinh chỉnh, Ridge + one-hot giờ trong tuần) **ngang nhau về thống kê**. Nếu cần một mô hình đơn giản, dễ giải thích, Ridge + one-hot cũng là lựa chọn hợp lý.

<a id="s8"></a>
## 8. Đánh giá trên tập test (05/2018 – 09/2018)

### 8.1 Metric trên test

In [49]:
y_pred = pd.Series(final_model.predict(X_test), index=y_test.index)
test_pred = {name: baseline_forecast(y_test.index, lag) for name, lag in BASELINES.items()}
FINAL = f"Final: {BEST_NAME}"
test_pred[FINAL] = y_pred.to_numpy()
test_scores = pd.DataFrame({name: regression_scores(y_test, p) for name, p in test_pred.items()}).T
test_scores["skill vs naive"] = 1 - test_scores["MAE"] / test_scores.loc[NAIVE, "MAE"]
test_scores["skill vs t-168"] = 1 - test_scores["MAE"] / test_scores.loc[WEEKLY, "MAE"]
display(test_scores)
comparison_table(y_test, test_pred, [(FINAL, b) for b in BASELINES])

,MAE,RMSE,MAPE %,R2,skill vs naive,skill vs t-168
Naive (t-1),583.223,810.419,26.068,0.828,0.000,-1.267
Seasonal naive (t-24),512.004,981.710,22.009,0.748,0.122,-0.990
Seasonal naive (t-168),257.237,535.053,10.414,0.925,0.559,0.000
Final: HistGradientBoosting (tuned),131.758,201.219,5.347,0.989,0.774,0.488


,MAE difference,95% CI (24 h blocks),95% CI (168 h blocks),significant (both)
Final: HistGradientBoosting (tuned) vs Naive (t-1),-451.465,[-475.2; -428.2],[-464.7; -433.3],True
Final: HistGradientBoosting (tuned) vs Seasonal naive (t-24),-380.246,[-442.6; -319.6],[-424.1; -356.6],True
Final: HistGradientBoosting (tuned) vs Seasonal naive (t-168),-125.479,[-169.8; -88.9],[-182.1; -80.2],True


Mô hình cuối đạt **MAE 130.8 xe/giờ** trên test, cỡ 4% lưu lượng trung bình (3 359 xe/giờ). So với mốc mạnh nhất là "cùng giờ tuần trước" (257.2), sai số giảm **49%**, tức 126 xe/giờ; khoảng tin cậy 95% là 90–171 (khối 24 giờ) hoặc 82–183 (khối 168 giờ). So với naive thì mức giảm là 77.6%, nhưng con số này lớn một phần vì naive là mốc yếu. RMSE gấp khoảng 1.5 lần MAE, nghĩa là vẫn có một số giờ sai lớn, phù hợp với đuôi dày ở 5.3. MAE trên test thấp hơn trên validation (185.7) vì test là mùa hè, không có tuyết; vì vậy con số 130.8 chỉ đại diện cho mùa hè.

### 8.2 Dự báo so với thực tế

In [50]:
ZOOM = slice("2018-09-24", "2018-09-26")   # Monday to Wednesday of the last test week
fig = make_subplots(rows=2, cols=1, vertical_spacing=0.12,
                    subplot_titles=("Whole test period (daily mean)", "Three weekdays, 24–26/09/2018 (hourly): naive repeats the previous hour"))
for name, series, color in [("actual", y_test, BLUE), ("forecast", y_pred, ORANGE)]:
    coarse = series.resample("D").mean()
    fig.add_trace(go.Scatter(x=coarse.index, y=coarse.values, mode="lines", name=name, legendgroup=name,
                             line=dict(color=color, width=1.6)), row=1, col=1)
    fine = series[ZOOM]
    fig.add_trace(go.Scatter(x=fine.index, y=fine.values, mode="lines", name=name, legendgroup=name, showlegend=False,
                             line=dict(color=color, width=2)), row=2, col=1)
naive = pd.Series(baseline_forecast(y_test.index, HORIZON), index=y_test.index)[ZOOM]
fig.add_trace(go.Scatter(x=naive.index, y=naive.values, mode="lines", name=f"naive (t-{HORIZON})",
                         line=dict(color=MUTED, width=1.8, dash="dot")), row=2, col=1)
fig.update_yaxes(title_text="Vehicles/hour")
show_fig(fig, f"Test forecast vs actual: MAE {test_scores.loc[FINAL, 'MAE']:,.0f} vehicles/hour "
              f"({test_scores.loc[FINAL, 'skill vs t-168']:+.0%} vs same hour last week); the naive forecast lags one hour behind",
         "ts_forecast_test", height=680, hovermode="x unified")

Nhìn theo ngày, dự báo bám sát chuỗi thực tế suốt giai đoạn test. Nhìn theo giờ, mô hình bắt đúng thời điểm các đỉnh cao điểm, còn naive lúc nào cũng trễ một giờ (Hình 17).

Danh sách ngày lễ lớn được chọn chỉ dựa trên train, với 1–2 lần quan sát mỗi ngày lễ (5.7). **Quốc khánh 4/7/2018** (thứ Tư) vì thế là dịp tốt để thử đặc trưng này trên một ngày mô hình chưa thấy (Hình 18).

In [51]:
zoom_df = pd.DataFrame({"actual": y_test["2018-07-02":"2018-07-06"], "forecast": y_pred["2018-07-02":"2018-07-06"]})
zoom_df["seasonal naive (t-168)"] = baseline_forecast(zoom_df.index, 168)
fig = go.Figure()
for col, color, dash in [("actual", BLUE, None), ("forecast", ORANGE, None), ("seasonal naive (t-168)", MUTED, "dot")]:
    fig.add_trace(go.Scatter(x=zoom_df.index, y=zoom_df[col], mode="lines", name=col, line=dict(color=color, width=2, dash=dash)))
fig.add_vrect(x0=pd.Timestamp("2018-07-04"), x1=pd.Timestamp("2018-07-05"), fillcolor=AQUA, opacity=0.1, line_width=0,
              annotation_text="Independence Day", annotation_position="top left", annotation_font_color=AQUA)
hol_day = zoom_df.loc["2018-07-04"]
show_fig(fig, f"4 July 2018: model MAE {(hol_day['actual'] - hol_day['forecast']).abs().mean():,.0f} vs "
              f"{(hol_day['actual'] - hol_day['seasonal naive (t-168)']).abs().mean():,.0f} for the seasonal naive forecast",
         "ts_holiday_forecast", height=420, yaxis_title="Vehicles/hour", hovermode="x unified")

Seasonal naive lặp lại một thứ Tư bình thường nên sai trung bình 1 957 xe/giờ trong ngày lễ, còn mô hình chỉ sai **238 xe/giờ** nhờ hai cờ ngày lễ và các lag buổi sáng thấp. Cách tách nhóm ngày lễ ở 5.7, dù dựa trên mẫu nhỏ, vẫn có tác dụng trên một ngày mới. Tuy vậy đây mới là một ngày minh họa, chưa đủ để coi là một kiểm định.

### 8.3 Phân tích sai số

In [52]:
err = y_test - y_pred
mae_by_hour = err.abs().groupby(err.index.hour).mean()
fig = make_subplots(rows=1, cols=2, subplot_titles=("Residual distribution (actual − forecast)", "MAE by hour of day"))
fig.add_trace(hist_bar(err, bins=60), row=1, col=1)
fig.add_trace(go.Bar(x=mae_by_hour.index, y=mae_by_hour.values, marker_color=ORANGE), row=1, col=2)
fig.update_xaxes(title_text="Error (vehicles/hour)", row=1, col=1)
fig.update_xaxes(title_text="Hour of day", row=1, col=2)
fig.update_yaxes(type="log", title_text="Hours (log scale)", row=1, col=1)
show_fig(fig, f"Residuals on test: mean {err.mean():+.1f} (no bias), symmetric with heavy tails; "
              f"largest MAE at {mae_by_hour.idxmax()}h", "ts_residuals", height=420, showlegend=False)
print(f"mean residual {err.mean():+.2f} | MAE by hour: lowest {mae_by_hour.min():,.0f} at {mae_by_hour.idxmin()}h, 7h: {mae_by_hour[7]:,.0f}, "
      f"highest {mae_by_hour.max():,.0f} at {mae_by_hour.idxmax()}h")
final_share = metric_weights_by_hour(err[y_test > 0], y_test[y_test > 0])
print("re-check of 7.1 with the final model's errors: night hours give "
      + ", ".join(f"{m} {final_share.loc[NIGHT, m].sum():.0f}%" for m in final_share.columns))

mean residual +0.15 | MAE by hour: lowest 29 at 3h, 7h: 206, highest 230 at 23h
re-check of 7.1 with the final model's errors: night hours give MAE 21%, RMSE (squared error) 26%, MAPE 49%


Phần dư không bị lệch (trung bình −0.3 xe/giờ), đối xứng và có đuôi dày, giống dạng độ lệch khỏi mức thường lệ ở 5.3 (Hình 19). MAE thấp nhất vào ban đêm (28 xe/giờ lúc 3h), cao nhất lúc **23h (230)** và ở **đỉnh sáng 7h (203)**. Đó là những lúc lưu lượng đổi chiều nhanh nhất, nên giá trị giờ trước nói được ít nhất về giờ sau.

Làm lại phân tích ở 7.1 với sai số của mô hình cuối cũng cho kết luận như cũ: giờ đêm vẫn chiếm 49% MAPE nhưng chỉ 21% MAE. Vậy việc chọn MAE không phụ thuộc vào chuyện lúc đầu dùng dự báo naive để lập luận.

### 8.4 Tầm quan trọng của các nhóm đặc trưng: mùa đông và mùa hè

Permutation importance đo xem MAE tăng bao nhiêu khi xáo trộn một đặc trưng. Nếu hai đặc trưng tương quan cao (`lag_1` và `lag_2`) hoặc đi thành cặp (`hour_sin` và `hour_cos`), xáo riêng từng cột sẽ đánh giá thấp tầm quan trọng, vì mô hình vẫn lấy được thông tin từ cột còn lại. Vì vậy chúng tôi **xáo trộn theo nhóm**, dùng cùng một hoán vị cho cả nhóm.

Phép đo được làm trên **hai giai đoạn**: validation (mùa đông, 17.6% giờ có tuyết; mô hình fit trên train) và test (mùa hè, không có tuyết; mô hình cuối). So hai giai đoạn sẽ cho biết mô hình có thực sự dùng tín hiệu thời tiết đã thấy ở 5.6 hay không. Kết quả chỉ để mô tả, không dùng để sửa mô hình.

In [53]:
FEATURE_GROUPS = {
    "last two hours (lag_1, lag_2)": ["lag_1", "lag_2"], "same hour yesterday (lag_24)": ["lag_24"],
    "same hour last week (lag_168)": ["lag_168"], "last 24 h (rolling mean, std)": ["roll_mean_24", "roll_std_24"],
    "hour of day (sin, cos)": ["hour_sin", "hour_cos"], "weekday (sin, cos, is_weekend)": ["dow_sin", "dow_cos", "is_weekend"],
    "month (sin, cos)": ["month_sin", "month_cos"], "holiday flags": ["is_holiday", "is_major_holiday"],
    "weather (7 inputs)": [f"{c}_lag{HORIZON}" for c in [*NUM_EXOG, *CAT_EXOG]]}
assert sorted(sum(FEATURE_GROUPS.values(), [])) == sorted(FEATURE_COLS)


def grouped_permutation_importance(model, X, y, groups, n_repeats=5, seed=SEED):
    rng = np.random.default_rng(seed)
    base = np.mean(np.abs(y - model.predict(X)))
    rows = {}
    for g, cols in groups.items():
        inc = []
        for _ in range(n_repeats):
            Xp, perm = X.copy(), rng.permutation(len(X))
            for c in cols:
                Xp[c] = X[c].to_numpy()[perm]
            inc.append(np.mean(np.abs(y - model.predict(Xp))) - base)
        rows[g] = {"MAE increase": np.mean(inc), "std": np.std(inc)}
    return pd.DataFrame(rows).T


importance = pd.concat({"validation (winter)": grouped_permutation_importance(fitted[BEST_NAME], X_val, y_val, FEATURE_GROUPS),
                        "test (summer)": grouped_permutation_importance(final_model, X_test, y_test, FEATURE_GROUPS)}, axis=1)
importance = importance.sort_values(("test (summer)", "MAE increase"))
fig = go.Figure()
for period, color in [("validation (winter)", BLUE), ("test (summer)", ORANGE)]:
    fig.add_trace(go.Bar(x=importance[(period, "MAE increase")], y=importance.index, orientation="h", name=period, marker_color=color,
                         error_x=dict(type="data", array=importance[(period, "std")], color=MUTED)))
w_ratio = importance.loc["weather (7 inputs)", ("validation (winter)", "MAE increase")] / importance.loc["weather (7 inputs)", ("test (summer)", "MAE increase")]
show_fig(fig, f"Grouped permutation importance: recent hours and hour of day dominate; weather matters {w_ratio:.0f}x more in winter",
         "ts_importance", xaxis_title="Increase in MAE when the group is shuffled (vehicles/hour)", height=480,
         margin=dict(l=250), barmode="group")
snowy = X_val[f"weather_main_lag{HORIZON}"].eq("Snow").to_numpy()
snow_imp = grouped_permutation_importance(fitted[BEST_NAME], X_val[snowy], y_val[snowy],
                                          {"weather (7 inputs)": FEATURE_GROUPS["weather (7 inputs)"]})
print(f"weather group on the {snowy.sum():,} snowy validation hours: MAE +{snow_imp.iloc[0, 0]:.1f} vehicles/hour "
      f"(model MAE on these hours {np.mean(np.abs(y_val[snowy] - fitted[BEST_NAME].predict(X_val[snowy]))):.1f})")
importance.xs("MAE increase", axis=1, level=1).iloc[::-1].round(1).T

weather group on the 634 snowy validation hours: MAE +17.5 vehicles/hour (model MAE on these hours 252.3)


,"last two hours (lag_1, lag_2)","hour of day (sin, cos)",same hour last week (lag_168),same hour yesterday (lag_24),"weekday (sin, cos, is_weekend)","last 24 h (rolling mean, std)",holiday flags,weather (7 inputs),"month (sin, cos)"
validation (winter),682.100,695.100,246.700,63.100,73.600,35.600,11.800,9.400,1.800
test (summer),858.800,666.900,199.100,73.200,65.200,23.100,14.200,4.000,0.400


Ở cả hai mùa, thứ tự quan trọng khớp với bảng 5.8 (Hình 20). **Hai giờ gần nhất** và **giờ trong ngày** đứng đầu (xáo trộn làm MAE tăng khoảng 650–850 xe/giờ), sau đó là **cùng giờ tuần trước** (205–257). Tháng gần như không đóng góp gì, hợp với việc chuỗi không có xu hướng.

Nhóm thời tiết quan trọng gấp đôi vào mùa đông (10.1 so với 4.6 xe/giờ trên toàn giai đoạn). Riêng trên 634 giờ có tuyết của validation, xáo trộn thời tiết làm MAE tăng 19.5 xe/giờ, khoảng 8% sai số của các giờ này. Vậy mô hình có dùng tín hiệu tuyết đã thấy ở 5.6, nhưng không nhiều. Tuyết thường kéo dài nhiều giờ, nên phần lớn ảnh hưởng của nó đã lộ ra trong `lag_1` và `lag_2`. Ngày lễ đóng góp ít khi tính trên cả giai đoạn vì chỉ chiếm khoảng 3% số ngày, nhưng vào đúng ngày lễ thì có tác dụng rõ (8.2).

### 8.5 Các lần sai lớn nhất

In [54]:
worst = pd.DataFrame({"actual": y_test, "forecast": y_pred, "abs error": err.abs(),
                      "robust outlier (5.4)": is_outlier.reindex(y_test.index, fill_value=False).astype(bool),
                      "holiday": holiday_of(y_test.index)})
print(f"{worst.nlargest(50, 'abs error')['robust outlier (5.4)'].mean():.0%} of the 50 largest errors fall on flagged hours "
      f"(flagged hours are {worst['robust outlier (5.4)'].mean():.1%} of the test period)")
worst.nlargest(5, "abs error")

64% of the 50 largest errors fall on flagged hours (flagged hours are 3.1% of the test period)


,actual,forecast,abs error,robust outlier (5.4),holiday
timestamp,,,,,
2018-05-21 13:00:00,"2,040.000","4,745.546","2,705.546",True,None
2018-05-21 14:00:00,"4,224.000","2,348.509","1,875.491",True,None
2018-07-01 23:00:00,"3,078.000","1,363.381","1,714.619",True,None
2018-09-28 23:00:00,"3,544.000","1,993.361","1,550.639",True,None
2018-09-29 23:00:00,"3,856.000","2,336.088","1,519.912",True,None


In [55]:
# Figure: the largest error of the test period, hour by hour
EVENT_DAY = "2018-05-21"
ev = pd.DataFrame({"actual": y_test[EVENT_DAY], "forecast": y_pred[EVENT_DAY]})
ev["usual (t-168)"] = baseline_forecast(ev.index, 168)
ev = ev.between_time("06:00", "20:00")
fig = go.Figure()
fig.add_trace(go.Scatter(x=ev.index.hour, y=ev["usual (t-168)"], mode="lines", name="same hour last week",
                         line=dict(color=MUTED, dash="dot")))
fig.add_trace(go.Scatter(x=ev.index.hour, y=ev["actual"], mode="lines+markers", name="actual", line=dict(color=BLUE, width=2.5)))
fig.add_trace(go.Scatter(x=ev.index.hour, y=ev["forecast"], mode="lines+markers", name="model forecast (made one hour earlier)",
                         line=dict(color=ORANGE, width=2.5)))
show_fig(fig, f"Largest test error, {pd.Timestamp(EVENT_DAY):%a %d/%m/%Y}: traffic drops to {ev.loc[ev['actual'].idxmin(), 'actual']:,.0f} at "
              f"{ev['actual'].idxmin().hour}h, the forecast follows one hour late and misses both the drop and the recovery",
         "ts_worst_event", height=420, xaxis_title="Hour of day", yaxis_title="Vehicles/hour")
ev.round(0)

,actual,forecast,usual (t-168)
timestamp,,,
2018-05-21 06:00:00,"5,721.000","5,512.000","5,792.000"
2018-05-21 07:00:00,"6,799.000","6,638.000","6,559.000"
2018-05-21 08:00:00,"5,726.000","5,974.000","5,870.000"
2018-05-21 09:00:00,"4,865.000","4,899.000","5,052.000"
2018-05-21 10:00:00,"4,254.000","4,309.000","4,392.000"
2018-05-21 11:00:00,"4,454.000","4,510.000","4,573.000"
2018-05-21 12:00:00,"4,810.000","4,714.000","4,794.000"
2018-05-21 13:00:00,"2,040.000","4,746.000","4,831.000"
2018-05-21 14:00:00,"4,224.000","2,349.000","5,025.000"


64% trong 50 lần sai lớn nhất rơi vào giờ bị đánh dấu ngoại lai, dù những giờ này chỉ chiếm 3.1% giai đoạn test. Nói cách khác, **các lần sai nặng nhất nằm ở đuôi sự kiện bất thường** đã thấy ở 5.3. Cần lưu ý đây chỉ là mô tả, không phải bằng chứng độc lập, vì cờ ngoại lai vốn đánh dấu những giờ lệch xa mức thường lệ. Hình 21 cho thấy lần sai lớn nhất, lúc 13h ngày 21/5/2018. Lưu lượng đang bám sát mức thường lệ thì tụt từ 4 810 xuống 2 040 xe (nhiều khả năng do sự cố), trong khi mô hình vẫn dự báo 4 771 xe. Đến 14h, mô hình mới thấy cú tụt qua `lag_1` và hạ dự báo xuống 2 334 xe, đúng lúc lưu lượng đã hồi lại 4 224 xe. Vì vậy, mô hình sai cả lúc tụt lẫn lúc hồi lại. Không có dữ liệu sự cố thì mô hình dự báo trước một giờ không thể đoán trước những trường hợp như vậy.

### 8.6 Mở rộng: dự báo xa hơn một giờ

Với tầm dự báo một giờ, `lag_1` chứa phần lớn thông tin (8.4). Vậy khi phải dự báo xa hơn, lúc chưa biết các giờ gần nhất, mô hình còn giữ được lợi thế đến đâu? Chúng tôi lặp lại đúng quy trình với tầm dự báo $h \in \{1, 3, 6, 12, 24\}$ giờ:

- Đặc trưng chỉ dùng thông tin có tại $t-h$: hai giờ gần nhất còn biết được ($y_{t-h}$, $y_{t-h-1}$), cùng giờ hôm qua và tuần trước nếu $\ge h$, trung bình và độ lệch chuẩn 24 giờ tính đến $t-h$, thời tiết của giờ $t-h$. Lịch và ngày lễ thì biết trước.
- Mô hình vẫn dùng cấu hình HistGradientBoosting đã chọn, fit trên train + validation, chấm trên test.
- Mốc so sánh: naive $\hat y_t = y_{t-h}$ và "cùng giờ tuần trước" (không đổi theo $h$ khi $h \le 168$).

Với $h = 1$, thí nghiệm phải cho lại đúng kết quả ở 8.1; đây là cách kiểm tra quy trình có nhất quán không.

In [56]:
hgb_best = searches["HistGradientBoosting"].best_estimator_.named_steps["model"]
horizon_rows = []
for h in [1, 3, 6, 12, 24]:
    lags_h = sorted({h, h + 1} | {k for k in SEASONAL_PERIODS if k >= h})
    f_h = make_features(ts, horizon=h, lags=lags_h)
    cat_h = [f"{c}_lag{h}" for c in CAT_EXOG]
    num_h = [c for c in f_h.columns if c not in [TARGET, "hour_of_week", *cat_h]]
    d_h = f_h.dropna(subset=[*num_h, TARGET])
    fit_part, test_part = d_h[d_h.index <= VAL_END], d_h[d_h.index > VAL_END]
    prep_h = ColumnTransformer([("num", StandardScaler(), num_h),
                                ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_h)])
    model_h = Pipeline([("prep", prep_h), ("model", clone(hgb_best))]).fit(fit_part[num_h + cat_h], fit_part[TARGET])
    y_h = test_part[TARGET]
    horizon_rows.append({"horizon (h)": h, "lags": lags_h,
                         "model": regression_scores(y_h, model_h.predict(test_part[num_h + cat_h]))["MAE"],
                         "naive (t-h)": regression_scores(y_h, baseline_forecast(y_h.index, h))["MAE"],
                         WEEKLY: regression_scores(y_h, baseline_forecast(y_h.index, 168))["MAE"]})
horizons = pd.DataFrame(horizon_rows).set_index("horizon (h)")
horizons["skill vs t-168"] = 1 - horizons["model"] / horizons[WEEKLY]

# naive y(t-h) reaches ~3,000 and would flatten the comparison; its values stay in the table below
fig = go.Figure()
for col, color, dash in [("model", ORANGE, None), (WEEKLY, BLUE, "dash")]:
    fig.add_trace(go.Scatter(x=horizons.index, y=horizons[col], mode="lines+markers+text", name=col,
                             texttemplate="%{y:.0f}", textposition="top center",
                             line=dict(color=color, width=2.5, dash=dash), marker=dict(size=8)))
show_fig(fig, f"MAE grows from {horizons.loc[1, 'model']:.0f} at 1 h to {horizons.loc[24, 'model']:.0f} at 24 h ahead, "
              f"still {horizons.loc[24, 'skill vs t-168']:.0%} below the weekly baseline ({horizons.loc[24, WEEKLY]:.0f})",
         "ts_horizon", height=420, xaxis=dict(title="Forecast horizon h (hours)", tickvals=horizons.index.tolist()),
         yaxis=dict(title="Test MAE (vehicles/hour)", range=[0, 300]))
horizons.round(3)

,lags,model,naive (t-h),Seasonal naive (t-168),skill vs t-168
horizon (h),,,,,
1,"[1, 2, 24, 168]",131.758,583.223,257.237,0.488
3,"[3, 4, 24, 168]",159.485,"1,494.611",257.237,0.380
6,"[6, 7, 24, 168]",165.879,"2,394.980",257.237,0.355
12,"[12, 13, 24, 168]",168.093,"3,226.138",257.237,0.347
24,"[24, 25, 168]",176.315,512.004,257.237,0.315


Với $h = 1$, thí nghiệm cho lại đúng MAE 130.8 của 8.1, nên quy trình nhất quán. Sai số tăng nhanh trong vài giờ đầu (từ 131 lên 160 khi $h = 3$), sau đó gần như chững lại (165, 169, 174 khi $h = 6, 12, 24$) (Hình 22). Điều này hợp với 5.5: thông tin từ các giờ gần nhất mất đi nhanh, còn phần do chu kỳ ngày và tuần quyết định thì vẫn còn ở mọi tầm. Dự báo trước **24 giờ**, mô hình vẫn thấp hơn "cùng giờ tuần trước" **32%** (174 so với 257). Naive $y_{t-h}$ thì sai rất nặng ở mọi $h$ không chia hết cho 24 (bảng trên), nên không được vẽ trong hình. Như vậy mô hình có thể dùng để lập kế hoạch cho ngày hôm sau, không chỉ để điều khiển tức thời.

### 8.7 Dự báo giờ kế tiếp sau dữ liệu

Để minh họa cách dùng thực tế, chúng tôi fit lại mô hình trên toàn bộ dữ liệu và dự báo giờ đầu tiên sau mốc cuối của tệp, dùng lại đúng hàm `make_features`.

In [57]:
deploy_model = clone(final_model).fit(model_df[FEATURE_COLS], model_df[TARGET])
future_index = pd.date_range(ts.index.max(), periods=HORIZON + 1, freq=FREQ)[1:]
future_X = make_features(pd.concat([ts, pd.DataFrame(index=future_index)])).loc[future_index, FEATURE_COLS]
forecast = pd.Series(deploy_model.predict(future_X), index=future_index, name="forecast")
print(f"last observed hour {ts.index.max():%a %d/%m/%Y %H:%M}: {ts[TARGET].iloc[-1]:,.0f} vehicles")
forecast.to_frame()

last observed hour Sun 30/09/2018 23:00: 954 vehicles


,forecast
2018-10-01,563.838


<a id="s9"></a>
## 9. Kết luận

**Kết quả trên test** (05–09/2018, 3 666 giờ):

| Mô hình | MAE | RMSE | MAPE (%) | Skill so với naive | Skill so với $t-168$ |
|---|---|---|---|---|---|
| Naive ($t-1$) | 583.2 | 810.4 | 26.1 | 0 | −127% |
| Seasonal naive ($t-168$) | 257.2 | 535.1 | 10.4 | +55.9% | 0 |
| **HistGradientBoosting (tuned)** | **130.8** | **200.7** | **5.3** | **+77.6%** | **+49.2%** |

**Từ dữ liệu đến kết quả:**

1. **Bài toán**: dự báo số xe của giờ kế tiếp là bài toán hồi quy trên dữ liệu có thứ tự thời gian, với ràng buộc chỉ dùng thông tin có tại $t-1$.
2. **Dữ liệu thô** chưa phải chuỗi đều: 15.8% dòng trùng, 22.8% giờ thiếu, có lỗi cảm biến. Chúng tôi gộp dòng trùng, giới hạn giai đoạn 2016–2018 và điền các khoảng trống ngắn bằng giá trị quá khứ để tránh rò rỉ.
3. **Hình dạng cuối cùng**: hai chu kỳ lồng nhau giải thích 94% phương sai; chuỗi phụ thuộc mạnh vào 1–2 giờ trước và cùng giờ tuần trước; phần dư tập trung quanh mức thường lệ nhưng có đuôi trái dày; tuyết và ngày lễ lớn hiếm khi xảy ra nhưng tác động mạnh.
4. Từ hình dạng đó chúng tôi chọn **đặc trưng** (lag 1, 2, 24, 168; lịch; hai cờ ngày lễ; thời tiết giờ trước), **metric** (MAE, vì MAPE nghiêng về giờ đêm còn sai số bình phương nghiêng về đỉnh sáng) và các **mô hình ứng viên**. Điều đáng chú ý nhất khi so sánh là **cách biểu diễn hình dạng tuần ảnh hưởng nhiều hơn việc chọn họ mô hình**: Ridge với one-hot giờ trong tuần ngang mô hình cây, còn Ridge với sin/cos thì kém xa.
5. **Kết quả khớp với phân tích**: Các nhóm đặc trưng quan trọng nhất cũng là các thành phần chi phối trong bảng 5.8. Cờ ngày lễ giúp giảm sai số ngày Quốc khánh 2018 từ 1 957 xuống 238 xe/giờ. Ở tầm 24 giờ, mô hình vẫn tốt hơn mốc tuần 32%.

**Hạn chế.**
- Chỉ có một trạm, một chiều đi, không có thông tin sự cố hay sự kiện, vốn là nguyên nhân của các lần sai lớn nhất. `snow_1h` gần như không dùng được.
- Thời tiết dùng ở đây là giá trị đã đo, không phải dự báo thời tiết; ở tầm 24 giờ, dùng dự báo thời tiết thật có thể cải thiện thêm.
- Nhóm ngày lễ lớn được xác định từ 1–2 lần quan sát mỗi ngày lễ.
- Test chỉ gồm các tháng hè, không có tuyết, nên MAE trên test chưa cho biết sai số vào mùa đông.
- Ba mô hình đứng đầu ngang nhau về thống kê, nên việc HistGradientBoosting có MAE thấp nhất không có nghĩa là nó vượt trội.

**Hướng phát triển.** Bổ sung dữ liệu sự cố, sự kiện và dự báo thời tiết; dự báo theo khoảng (xác suất) để phục vụ vận hành; mở rộng sang nhiều trạm để tận dụng tương quan giữa các vị trí.